Previous script trained multiple models using different ML protocols; this script evaluates models from across different protocols (which otherwise can't be directly compared) to select a final one to use for feature generation.

TEST "TYPES":
- **Stability** = consistency of inferred state structure
- **Generalization** = out-of-sample explanatory power

See other markup fields for more details re: specific testing procedures and evaluation metrics.

[Runtime: Dependent on number of models, but usually at least 10-15 min]

---------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, subprocess
import time
from pathlib import Path
import json
import pandas as pd
import numpy as np
import joblib
from sklearn.decomposition import PCA
from hmmlearn.hmm import GaussianHMM

In [ ]:
# __________________________________________________________________________________________________________
### LOAD PARAMETERS:

# General parameters:
HARD_STOP    = config['hard_errors']
RANDOM_SEED  = config['random_seed']

OVERWRITE_FINAL_MODEL = bool(config["cross_model_evaluation"].get("overwrite_final_model", False))

# I think we still need these, e.g. to grab the "winning" protocol parameters for the last re-fitting on X_all after the last round of model-selection:
HMM_DEFAULTS = config["HMM_protocols"]["defaults"]
HMM_PRESETS  = config["HMM_protocols"]["presets"]


# Cross-model evaluation parameters:
CROSS_MODEL_EVAL = bool(config["cross_model_evaluation"]["enabled"])
SPLIT_UNIT = str(config["cross_model_evaluation"]["split_unit"]).strip().lower()

# Keep the remaining first-level blocks as dictionaries for later use:
ALIGN_STATES_CFG = dict(config["cross_model_evaluation"]["align_states"])
MINIMUMS_CFG = dict(config["cross_model_evaluation"]["minimums"])
STABILITY_CFG = dict(config["cross_model_evaluation"]["stability"])
GENERALIZATION_CFG = dict(config["cross_model_evaluation"]["generalization"])

# Light normalization (keeps downstream code simpler/consistent)
ALIGN_STATES_CFG["method"] = str(ALIGN_STATES_CFG.get("method", "hungarian")).strip().lower()
ALIGN_STATES_CFG["distance"] = str(ALIGN_STATES_CFG.get("distance", "means-l2")).strip().lower()
STABILITY_CFG["split_strategy"] = str(STABILITY_CFG.get("split_strategy", "blocked_random")).strip().lower()
STABILITY_CFG["mode"] = str(STABILITY_CFG.get("mode", "fixed_model")).strip().lower()
# Minimal essential validation
if SPLIT_UNIT not in {"subject", "sequence"}:
    raise ValueError(f"[INIT ERROR] cross_model_evaluation.split_unit must be 'subject' or 'sequence' (got: '{SPLIT_UNIT}')")
if ALIGN_STATES_CFG["method"] != "hungarian":
    raise ValueError(f"[INIT ERROR] cross_model_evaluation.align_states.method currently supports only 'hungarian' (got: '{ALIGN_STATES_CFG['method']}')")
if ALIGN_STATES_CFG["distance"] != "means-l2":
    raise ValueError(f"[INIT ERROR] cross_model_evaluation.align_states.distance currently supports only 'means-l2' (got: '{ALIGN_STATES_CFG['distance']}')")
if STABILITY_CFG["mode"] not in {"fixed_model", "refit_models"}:
    raise ValueError(f"[INIT ERROR] cross_model_evaluation.stability.mode must be 'fixed_model' or 'refit_models' (got: '{STABILITY_CFG['mode']}')")


# __________________________________________________________________________________________________________
### SET FILEPATHS:

BASE_DIRECTORY = Path(config['root_output_directory'])

RUN_MANIFEST_PATH    = BASE_DIRECTORY / 'subject_manifest.csv'
MEG_PARAMETERS_PATH = BASE_DIRECTORY / 'MEG_manifest.csv'


### INPUTS:

# Grab dataset-selection config variables:
DATASET_SELECTOR    = str(config["ML_training"]["dataset_selector"]).strip().lower()
DATASET_MANUAL_PATH = config["ML_training"].get("dataset_path", None)

# Original "dataset pointer" is in this directory, and it also contains the raw data ('X_train', 'X_all', etc.):
DATA_DIR = Path(BASE_DIRECTORY) / config['ML_prep']['training_data_dir']

# Actual models live here (under a directory w/ the same name as governs the pointer logic):
MODELS_DIR = Path(BASE_DIRECTORY) / config['HMM_training']['HMM_model_directory']

# [[[See next cell for actual input routing]]]


### OUTPUTS:
HMM_MODEL_DIRECTORY = config["HMM_training"]["HMM_model_directory"]
OUTPUT_DIR = Path(BASE_DIRECTORY) / HMM_MODEL_DIRECTORY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# __________________________________________________________________________________________________________
### INITIALIZATION:

RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
MEG_runs = pd.read_csv(MEG_PARAMETERS_PATH)
if "MEG_session_ID" in MEG_runs.columns:
    MEG_runs = MEG_runs.rename(columns={"MEG_session_ID": "session_ID"})


# Current active settings summary print:
print("\n[INIT] Active cross-model evaluation CONFIG settings:")
print(f"   enabled      = {CROSS_MODEL_EVAL}")
print(f"   split_unit   = '{SPLIT_UNIT}'")
print(f"   stability    = enabled={bool(STABILITY_CFG.get('enabled', False))}, mode='{STABILITY_CFG['mode']}', strategy='{STABILITY_CFG['split_strategy']}'")
print(f"   generalize   = enabled={bool(GENERALIZATION_CFG.get('enabled', False))}, n_folds={GENERALIZATION_CFG.get('n_folds', None)}, n_repeats={GENERALIZATION_CFG.get('n_repeats', None)}")

In [ ]:
# __________________________________________________________________________________________________________
### RESOLVE DATASET (X_all) + SCAN SELECTED MODELS (chosen_model.json + final-model_k*.joblib)
#
# Creates:
#   - DATASET_NAME              : str (e.g., "n-193_Craddock_50-ROIs_t-18(50)_STD")
#   - DATASET_DIR               : Path to frozen data dir (contains X_all.csv, provenance.json)
#   - DATASET_MODELS_DIR        : Path to model outputs dir for same dataset (contains protocol subdirs)
#   - x_all_path                : Path to X_all.csv
#   - provenance_path           : Path to provenance.json
#   - SCANNED_PROTOCOLS         : dict[protocol_name] -> dict(paths + parsed chosen_model.json)
#   - models_index_df           : DataFrame summary (one row per protocol found)
# __________________________________________________________________________________________________________

def handle_error(message: str):
    if HARD_STOP:
        raise RuntimeError(message)
    else:
        print(f"[WARN] {message}")

# Resolve 'DATASET_NAME' (label) and 'DATASET_DIR':
DATASET_NAME = None
DATASET_DIR = None

if DATASET_SELECTOR == "latest":
    pointer_path = Path(DATA_DIR) / "LATEST_DATASET.json"
    if not pointer_path.exists():
        raise FileNotFoundError(
            f"[INIT ERROR] dataset_selector='latest' but pointer file not found:\n  {pointer_path}")
    with open(pointer_path, "r") as f:
        pointer = json.load(f)
    # Pointer contains a full dataset_dir, e.g. '.../ML_training_data/<DATASET_NAME>/':
    pointer_dataset_dir = Path(pointer.get("dataset_dir", "")).expanduser()
    if pointer_dataset_dir is None or str(pointer_dataset_dir).strip() == "":
        raise RuntimeError(
            f"[INIT ERROR] Pointer file exists but missing/empty 'dataset_dir':\n  {pointer_path}")
    DATASET_NAME = pointer_dataset_dir.name
    DATASET_DIR = Path(DATA_DIR) / DATASET_NAME  # <-- enforce rooting under 'DATA_DIR'
elif DATASET_SELECTOR == "manual":
    if DATASET_MANUAL_PATH is None or str(DATASET_MANUAL_PATH).strip() == "":
        raise ValueError(
            "[INIT ERROR] dataset_selector='manual' but ML_training.dataset_path is empty.")
    manual_dir = Path(str(DATASET_MANUAL_PATH)).expanduser()
    DATASET_NAME = manual_dir.name  # <-- the subfolder name identifies the dataset
    DATASET_DIR = Path(DATA_DIR) / DATASET_NAME  # <-- enforces rooting under 'DATA_DIR'
else:
    raise ValueError(
        f"[INIT ERROR] ML_training.dataset_selector must be 'latest' or 'manual' "
        f"(got: {DATASET_SELECTOR})")

# Resolve required frozen-data inputs ('X_all' + provenance sidecar):
x_all_path = DATASET_DIR / "X_all.csv"
provenance_path = DATASET_DIR / "provenance.json"

missing_data_files = []
if not DATASET_DIR.exists():
    missing_data_files.append(f"DATASET_DIR (folder): {DATASET_DIR}")
if not x_all_path.exists():
    missing_data_files.append(f"X_all.csv: {x_all_path}")
if not provenance_path.exists():
    missing_data_files.append(f"provenance.json: {provenance_path}")

if missing_data_files:
    missing_str = "\n".join(f"  - {item}" for item in missing_data_files)
    raise FileNotFoundError(
        "[INIT ERROR] Could not locate required frozen dataset artifacts:\n"
        f"{missing_str}")

# Resolve corresponding models directory for the SAME dataset label:
DATASET_MODELS_DIR = Path(MODELS_DIR) / DATASET_NAME

if not DATASET_MODELS_DIR.exists():
    raise FileNotFoundError(
        "[INIT ERROR] Expected dataset-specific models directory not found.\n"
        f"  MODELS_DIR          = {MODELS_DIR}\n"
        f"  DATASET_NAME        = {DATASET_NAME}\n"
        f"  Expected models dir = {DATASET_MODELS_DIR}\n"
        "This usually means the training script did not run for this dataset, or the model directory name differs.")

# Scan protocol subdirs for selected models:
SCANNED_PROTOCOLS = {}

protocol_dirs = sorted([p for p in DATASET_MODELS_DIR.iterdir() if p.is_dir()])

for proto_dir in protocol_dirs:
    proto_name = proto_dir.name

    chosen_json = proto_dir / "chosen_model.json"

    # "final" artifacts (may be multiple if user ran multiple Ks historically):
    final_models = sorted(proto_dir.glob("final-model_k*.joblib"))

    # PCA artifacts (optional; record whatever exists):
    pca_train = proto_dir / "pca_fit_on_X_train.joblib"
    pca_all   = proto_dir / "pca_refit_on_X_all.joblib"

    # We only consider a protocol "selected" if it has 'chosen_model.json' AND at least one final model:
    if (not chosen_json.exists()) or (len(final_models) == 0):
        continue

    # Parse 'chosen_model.json' (best effort; do not hard-fail unless HARD_STOP):
    chosen_payload = None
    try:
        with open(chosen_json, "r") as f:
            chosen_payload = json.load(f)
    except Exception as exc:
        handle_error(f"[INIT WARN] Failed to parse chosen_model.json for protocol '{proto_name}': {exc}")

    # Determine which final model is the “current” one.
    # Preference order:
    #   1) Use chosen_payload['refit_model_path'] if present and exists;
    #   2) Else use chosen_payload['chosen_K'] to locate final-model_k{K}.joblib;
    #   3) Else fall back to the newest final-model_k*.joblib by modification timestamp ('mtime').
    final_model_path = None

    if isinstance(chosen_payload, dict):
        refit_path = chosen_payload.get("refit_model_path", None)
        if refit_path and Path(refit_path).exists():
            final_model_path = Path(refit_path)

    if final_model_path is None and isinstance(chosen_payload, dict):
        k = chosen_payload.get("chosen_K", None)
        if k is not None:
            candidate = proto_dir / f"final-model_k{int(k)}.joblib"
            if candidate.exists():
                final_model_path = candidate

    if final_model_path is None:
        # fallback: use newest model by modification time:
        final_model_path = sorted(final_models, key=lambda p: p.stat().st_mtime, reverse=True)[0]

    SCANNED_PROTOCOLS[proto_name] = {
        "protocol": proto_name,
        "protocol_dir": str(proto_dir),
        "chosen_json_path": str(chosen_json),
        "final_model_path": str(final_model_path),
        "all_final_models_found": [str(p) for p in final_models],
        "pca_fit_on_X_train_path": str(pca_train) if pca_train.exists() else None,
        "pca_refit_on_X_all_path": str(pca_all) if pca_all.exists() else None,
        "chosen_payload": chosen_payload}

if not SCANNED_PROTOCOLS:
    raise RuntimeError(
        "[INIT ERROR] No selected protocol directories were found.\n"
        f"Scanned: {DATASET_MODELS_DIR}\n"
        "Expected each protocol folder to contain BOTH:\n"
        "  - chosen_model.json\n"
        "  - final-model_k{K}.joblib\n")

# Build compact review table:
rows = []
for proto, info in SCANNED_PROTOCOLS.items():
    payload = info.get("chosen_payload", {}) if isinstance(info.get("chosen_payload", None), dict) else {}
    rows.append({
        "protocol": proto,
        "protocol_dir": info["protocol_dir"],
        "chosen_K": payload.get("chosen_K", None),
        "criterion_used": payload.get("metric_column_used", payload.get("criterion_used", None)),
        "final_model_path": info["final_model_path"],
        "pca_fit_on_X_train": info["pca_fit_on_X_train_path"] is not None,
        "pca_refit_on_X_all": info["pca_refit_on_X_all_path"] is not None,
        "n_final_models_found": len(info["all_final_models_found"])})

models_index_df = pd.DataFrame(rows).sort_values(["protocol"]).reset_index(drop=True)

# Print reports:
print("\n[INIT] Dataset + models resolved for arbitration:")
print(f"   DATA_DIR            = {DATA_DIR}")
print(f"   MODELS_DIR          = {MODELS_DIR}")
print(f"   DATASET_SELECTOR    = '{DATASET_SELECTOR}'")
print(f"   DATASET_NAME        = {DATASET_NAME}")
print(f"   DATASET_DIR         = {DATASET_DIR}")
print(f"     X_all.csv         = {x_all_path}")
print(f"     provenance.json   = {provenance_path}")
print(f"   DATASET_MODELS_DIR  = {DATASET_MODELS_DIR}")
print(f"   Protocols found     = {len(SCANNED_PROTOCOLS)}")

print("\n[INIT] Selected protocol artifacts discovered:")
display(models_index_df)

In [ ]:
# __________________________________________________________________________________________________________
### LOAD X_all + RECONSTRUCT (X_all, lengths_all) USING provenance.json FEATURE ORDER
#
# Requires previous locator cell outputs:
#   - x_all_path
#   - provenance_path
#
# Creates:
#   - X_all_df
#   - id_columns                (from provenance if available; else defaults)
#   - feature_columns_final      (from provenance if available; else inferred)
#   - X_all, lengths_all, sequences_all_df
# __________________________________________________________________________________________________________

# Read provenance (for column order):
provenance = None
id_columns = ["subject_ID", "session_ID", "time_index"]
feature_columns_final = None

try:
    with open(provenance_path, "r") as f:
        provenance = json.load(f)
except Exception as exc:
    handle_error(f"[ARBITRATION INIT] Could not read provenance.json; will infer columns from X_all.csv. Error: {exc}")
    provenance = None

if isinstance(provenance, dict):
    if "id_columns" in provenance and isinstance(provenance["id_columns"], list) and provenance["id_columns"]:
        id_columns = list(provenance["id_columns"])
    if "feature_columns" in provenance and isinstance(provenance["feature_columns"], list) and provenance["feature_columns"]:
        feature_columns_final = list(provenance["feature_columns"])

# Load 'X_all.csv':
try:
    X_all_df = pd.read_csv(x_all_path)
except Exception as exc:
    raise RuntimeError(f"[ARBITRATION INIT] Failed to read X_all.csv: {exc}")

# Sanity-check ID columns:
missing_ids = [c for c in id_columns if c not in X_all_df.columns]
if missing_ids:
    handle_error(f"[ARBITRATION INIT] X_all.csv missing required ID column(s): {missing_ids}")

# Infer feature columns (if provenance was missing / incomplete):
if feature_columns_final is None:
    feature_columns_final = [c for c in X_all_df.columns if c not in id_columns]
    feature_columns_final = sorted(feature_columns_final)

if not feature_columns_final:
    raise RuntimeError("[ARBITRATION INIT] No feature columns found in X_all.csv (only ID columns present).")

# Verify features exist:
missing_feats = [c for c in feature_columns_final if c not in X_all_df.columns]
if missing_feats:
    handle_error(f"[ARBITRATION INIT] X_all.csv missing feature column(s) from feature_columns_final: {missing_feats[:10]}")

# Build (X_all, lengths_all, sequences_all_df):
def build_X_and_lengths_from_table(df: pd.DataFrame, feature_cols: list) -> tuple:
    """
    Reconstruct sequence structure by grouping rows by (subject_ID, session_ID) after stable sorting.
    Returns:
      X (np.ndarray), lengths (list[int]), sequences_df (pd.DataFrame)
    """
    if df.empty:
        X = np.empty((0, len(feature_cols)), dtype=float)
        lengths = []
        sequences_df = pd.DataFrame(columns=["subject_ID", "session_ID", "length", "row_start", "row_end"])
        return X, lengths, sequences_df

    df_sorted = df.sort_values(["subject_ID", "session_ID", "time_index"]).reset_index(drop=True)

    seq_rows = []
    row_cursor = 0
    for (subj, sess), g in df_sorted.groupby(["subject_ID", "session_ID"], sort=False):
        L = int(g.shape[0])
        if L <= 0:
            continue
        seq_rows.append({
            "subject_ID": str(subj),
            "session_ID": str(sess),
            "length": L,
            "row_start": int(row_cursor),
            "row_end": int(row_cursor + L - 1)})
        row_cursor += L

    sequences_df = pd.DataFrame(seq_rows)
    if sequences_df.empty:
        X = np.empty((0, len(feature_cols)), dtype=float)
        lengths = []
        return X, lengths, sequences_df

    X = df_sorted[feature_cols].to_numpy(dtype=float)
    lengths = sequences_df["length"].astype(int).tolist()
    return X, lengths, sequences_df

X_all, lengths_all, sequences_all_df = build_X_and_lengths_from_table(X_all_df, feature_columns_final)

if X_all.shape[0] == 0 or len(lengths_all) == 0:
    raise RuntimeError("[ARBITRATION INIT] X_all is empty after reconstruction; cannot proceed with arbitration.")

if int(np.sum(lengths_all)) != int(X_all.shape[0]):
    handle_error(
        "[ARBITRATION INIT] Sum(lengths_all) does not match X_all rows. "
        f"sum(lengths_all)={int(np.sum(lengths_all))}, X_all.shape[0]={int(X_all.shape[0])}")

print("\n[ARBITRATION INIT] Loaded X_all and reconstructed sequences:")
print(f"   X_all shape        = {X_all.shape}  (rows = total timepoints, cols = features)")
print(f"   # sequences        = {len(lengths_all)}")
print(f"   sequence lengths   = (min={min(lengths_all)}, max={max(lengths_all)})")
print(f"   # features         = {len(feature_columns_final)}")

# print("\n[ARBITRATION INIT] Example sequences:")
# display(sequences_all_df[["subject_ID", "session_ID", "length"]].head(5))

----------

Next we want to set up the actual split-testing / K-fold data splits:

In [ ]:
# __________________________________________________________________________________________________________
### BUILD SPLIT UNITS TABLE (subject-level or sequence-level)
#
# Uses:
#   - SPLIT_UNIT (from YAML)
#   - sequences_all_df, lengths_all
#
# Creates:
#   - UNITS_DF     : one row per split unit
#   - SEQ_TO_UNIT  : dict[sequence_index] -> unit_id
# __________________________________________________________________________________________________________

def _as_int(x, default=None):
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return default
        return int(x)
    except Exception:
        return default

# Minimums (kept as INTs):
MIN_SEQS_PER_FOLD = _as_int(MINIMUMS_CFG.get("min_sequences_per_fold", 10), 10)
MIN_OBS_PER_FOLD  = _as_int(MINIMUMS_CFG.get("min_total_observations_per_fold", 500), 500)
MIN_SEQ_LEN       = _as_int(MINIMUMS_CFG.get("min_sequence_length", 10), 10)

# Ensure 'sequence_index' exists:
if "sequence_index" not in sequences_all_df.columns:
    sequences_all_df = sequences_all_df.copy()
    sequences_all_df["sequence_index"] = sequences_all_df.index.astype(int)

# Filter out too-short sequences (optional but usually correct):
seq_ok_mask = sequences_all_df["length"].astype(int) >= int(MIN_SEQ_LEN)
if int(seq_ok_mask.sum()) < len(sequences_all_df):
    msg = f"[CROSS-EVAL INIT] Dropping {len(sequences_all_df) - int(seq_ok_mask.sum())} sequences with length < {MIN_SEQ_LEN}."
    if HARD_STOP:
        print(msg)
    else:
        print("[WARN]", msg)

SEQUENCES_OK = sequences_all_df.loc[seq_ok_mask].reset_index(drop=True)

# Build 'UNITS_DF':
rows = []
SEQ_TO_UNIT = {}

if SPLIT_UNIT == "sequence":
    for _, r in SEQUENCES_OK.iterrows():
        seq_idx = int(r["sequence_index"])
        unit_id = f"{str(r['subject_ID'])}__{str(r['session_ID'])}"
        rows.append({
            "unit_id": unit_id,
            "subject_ID": str(r["subject_ID"]),
            "sequence_indices": [seq_idx],
            "n_sequences": 1,
            "n_observations": int(r["length"])})
        SEQ_TO_UNIT[seq_idx] = unit_id

elif SPLIT_UNIT == "subject":
    for subj, g in SEQUENCES_OK.groupby("subject_ID", sort=False):
        seq_indices = [int(x) for x in g["sequence_index"].tolist()]
        unit_id = str(subj)
        rows.append({
            "unit_id": unit_id,
            "subject_ID": str(subj),
            "sequence_indices": seq_indices,
            "n_sequences": int(len(seq_indices)),
            "n_observations": int(g["length"].astype(int).sum())})
        for seq_idx in seq_indices:
            SEQ_TO_UNIT[seq_idx] = unit_id

else:
    raise ValueError(f"[CROSS-EVAL INIT] SPLIT_UNIT must be 'subject' or 'sequence' (got: '{SPLIT_UNIT}')")

UNITS_DF = pd.DataFrame(rows)

if UNITS_DF.empty:
    raise RuntimeError("[CROSS-EVAL INIT] UNITS_DF is empty after construction; cannot proceed.")

print("\n[CROSS-EVAL INIT] Built split units:")
print(f"   SPLIT_UNIT             = '{SPLIT_UNIT}'")
print(f"   # units                = {UNITS_DF.shape[0]}")
print(f"   # sequences retained    = {SEQUENCES_OK.shape[0]}")
print(f"   unit obs (min/median/max)= ({UNITS_DF['n_observations'].min()}/{int(UNITS_DF['n_observations'].median())}/{UNITS_DF['n_observations'].max()})")
print(f"   unit seqs (min/median/max)= ({UNITS_DF['n_sequences'].min()}/{int(UNITS_DF['n_sequences'].median())}/{UNITS_DF['n_sequences'].max()})")

In [ ]:
# __________________________________________________________________________________________________________
### BUILD SPLITS FOR:
#   (1) STABILITY  (repeated half-splits)
#   (2) GENERALIZATION (K-fold CV, repeated)
#
# Requires:
#   - UNITS_DF with columns: unit_id, subject_ID, sequence_indices, n_sequences, n_observations
#   - SPLIT_UNIT
#   - STABILITY_CFG, GENERALIZATION_CFG, MINIMUMS_CFG
#   - RUN_MANIFEST (optional; only used if stratify_by_group=True)
#
# Creates:
#   - STABILITY_SPLITS : list[dict]
#   - CV_SPLITS        : list[dict]
# __________________________________________________________________________________________________________

if not CROSS_MODEL_EVAL:
    print("[CROSS-EVAL] cross_model_evaluation.enabled=False; skipping split generation.")
    STABILITY_SPLITS = []
    CV_SPLITS = []
    stab_enabled = bool(STABILITY_CFG.get("enabled", False))
    gen_enabled  = bool(GENERALIZATION_CFG.get("enabled", False))
else:
    # If config['random_seed'] is a valid int --> deterministic; if not provided / not int --> truly random:
    if isinstance(RANDOM_SEED, (int, np.integer)):
        rng = np.random.RandomState(int(RANDOM_SEED))
        print(f"[CROSS-EVAL SPLIT] Using deterministic RNG with random_seed={int(RANDOM_SEED)}")
    else:
        rng = np.random.RandomState(None)  # <-- assigns nondeterministic seed from OS entropy
        print("[CROSS-EVAL SPLIT] Using non-deterministic RNG (no valid random_seed provided)")

    # Set minimums (fold-level):
    MIN_SEQS_PER_FOLD = int(MINIMUMS_CFG.get("min_sequences_per_fold", 10))
    MIN_OBS_PER_FOLD  = int(MINIMUMS_CFG.get("min_total_observations_per_fold", 500))

    def _summarize_units(unit_ids):
        """Return (n_units, n_sequences, n_observations) for a set of unit_ids."""
        if unit_ids is None or len(unit_ids) == 0:
            return 0, 0, 0
        sub = UNITS_DF.loc[UNITS_DF["unit_id"].isin(unit_ids)]
        return (
            int(sub.shape[0]),
            int(sub["n_sequences"].astype(int).sum()),
            int(sub["n_observations"].astype(int).sum()))

    def _check_fold_minimums(unit_ids, label):
        n_units, n_seqs, n_obs = _summarize_units(unit_ids)
        if n_seqs < MIN_SEQS_PER_FOLD:
            handle_error(
                f"[CROSS-EVAL SPLIT] {label}: too few sequences (n_sequences={n_seqs} < {MIN_SEQS_PER_FOLD}).")
        if n_obs < MIN_OBS_PER_FOLD:
            handle_error(
                f"[CROSS-EVAL SPLIT] {label}: too few observations (n_observations={n_obs} < {MIN_OBS_PER_FOLD}).")
        return n_units, n_seqs, n_obs

    # (1) STABILITY SPLITS:
    STABILITY_SPLITS = []
    stab_enabled = bool(STABILITY_CFG.get("enabled", True))
    stab_strategy = str(STABILITY_CFG.get("split_strategy", "blocked_random")).strip()
    stab_repeats  = int(STABILITY_CFG.get("n_repeats", 20))
    stab_block_len = int(STABILITY_CFG.get("block_len", 2))

    unit_ids_all = UNITS_DF["unit_id"].astype(str).tolist()

    if stab_enabled:
        print("\n[CROSS-EVAL SPLIT] Building stability splits:")
        print(f"   strategy = {stab_strategy}")
        print(f"   n_repeats = {stab_repeats}")
        print(f"   block_len = {stab_block_len} (used only by blocked/odd-even strategies)")

        # Helper: build blocks of units (NOT timepoints) to reduce leakage / increase stability of splits.
        #    For SPLIT_UNIT = 'subject', a "block" is just a small group of subjects;
        #    For SPLIT_UNIT = 'sequence', a "block" is a small group of sequences.
        def _make_blocks(shuffled_ids, block_len):
            blocks = []
            for i in range(0, len(shuffled_ids), block_len):
                blocks.append(shuffled_ids[i:i+block_len])
            return blocks

        for r in range(stab_repeats):

            if stab_strategy == "contiguous_halves":
                # Stable ordering: use current 'UNITS_DF' order (already stable) or shuffle once per repeat if desired:
                ids = unit_ids_all.copy()
                # For contiguous halves, it’s usually better NOT to shuffle (represents “first half vs second half”):
                mid = len(ids) // 2
                A_ids = ids[:mid]
                B_ids = ids[mid:]

            elif stab_strategy == "odd_even_blocks":
                ids = unit_ids_all.copy()
                rng.shuffle(ids)
                blocks = _make_blocks(ids, stab_block_len)
                A_ids = [u for i, blk in enumerate(blocks) if (i % 2 == 0) for u in blk]
                B_ids = [u for i, blk in enumerate(blocks) if (i % 2 == 1) for u in blk]

            elif stab_strategy == "blocked_random":
                ids = unit_ids_all.copy()
                rng.shuffle(ids)
                blocks = _make_blocks(ids, stab_block_len)
                rng.shuffle(blocks)
                mid = len(blocks) // 2
                A_ids = [u for blk in blocks[:mid] for u in blk]
                B_ids = [u for blk in blocks[mid:] for u in blk]

            else:
                raise ValueError(
                    f"[CROSS-EVAL SPLIT] stability.split_strategy must be one of "
                    f"['blocked_random','odd_even_blocks','contiguous_halves'] (got: '{stab_strategy}')")

            # Fold minimum checks:
            _check_fold_minimums(A_ids, label=f"STABILITY repeat={r} split=A")
            _check_fold_minimums(B_ids, label=f"STABILITY repeat={r} split=B")

            STABILITY_SPLITS.append({
                "repeat": int(r),
                "A_unit_ids": A_ids,
                "B_unit_ids": B_ids,
                "strategy": stab_strategy,
                "block_len": int(stab_block_len)})

        print(f"[CROSS-EVAL SPLIT] Built {len(STABILITY_SPLITS)} stability split(s).")
    else:
        print("\n[CROSS-EVAL SPLIT] Stability testing disabled (stability.enabled=False).")

    # (2) GENERALIZATION SPLITS (K-fold CV):
    CV_SPLITS = []
    gen_enabled = bool(GENERALIZATION_CFG.get("enabled", True))
    n_folds = int(GENERALIZATION_CFG.get("n_folds", 5))
    n_repeats = int(GENERALIZATION_CFG.get("n_repeats", 1))
    stratify = bool(GENERALIZATION_CFG.get("stratify_by_group", False))

    # Build unit --> group label mapping (if stratification is enabled):
    UNIT_GROUP = None
    if gen_enabled and stratify:
        if RUN_MANIFEST is None or RUN_MANIFEST.empty:
            handle_error("[CROSS-EVAL SPLIT] stratify_by_group=True but RUN_MANIFEST is missing/empty. Disabling stratification.")
            stratify = False
        elif "group_ID" not in RUN_MANIFEST.columns:
            handle_error("[CROSS-EVAL SPLIT] stratify_by_group=True but RUN_MANIFEST lacks 'group_ID'. Disabling stratification.")
            stratify = False
        else:
            # Map subject_ID --> group_ID, then unit_id --> group_ID, via 'UNITS_DF.subject_ID':
            subj_to_group = (
                RUN_MANIFEST[["subject_ID", "group_ID"]]
                .dropna()
                .astype(str)
                .drop_duplicates(subset=["subject_ID"])
                .set_index("subject_ID")["group_ID"]
                .to_dict())
            UNIT_GROUP = {}
            for _, r in UNITS_DF.iterrows():
                u = str(r["unit_id"])
                s = str(r["subject_ID"])
                UNIT_GROUP[u] = subj_to_group.get(s, "UNKNOWN")

    if gen_enabled:
        if n_folds < 2:
            raise ValueError(f"[CROSS-EVAL SPLIT] generalization.n_folds must be >= 2 (got: {n_folds})")
        if n_folds > len(unit_ids_all):
            handle_error(
                f"[CROSS-EVAL SPLIT] generalization.n_folds={n_folds} > #units={len(unit_ids_all)}. "
                "Reducing folds to #units.")
            n_folds = len(unit_ids_all)

        print("\n[CROSS-EVAL SPLIT] Building generalization CV splits:")
        print(f"   n_folds   = {n_folds}")
        print(f"   n_repeats = {n_repeats}")
        print(f"   stratify_by_group = {stratify}")

        def _kfold_indices(ids, k):
            """Simple k-fold splitter over a list of ids. Returns list of folds (each fold is list of ids)."""
            ids = ids.copy()
            rng.shuffle(ids)
            folds = [[] for _ in range(k)]
            for i, u in enumerate(ids):
                folds[i % k].append(u)
            return folds

        def _kfold_indices_stratified(ids, k, unit_group_map):
            """
            Stratified k-fold over unit IDs using group labels.
            Greedy round-robin assignment within each group.
            """
            # Bucket by group:
            buckets = {}
            for u in ids:
                g = unit_group_map.get(u, "UNKNOWN")
                buckets.setdefault(g, []).append(u)

            # Shuffle each bucket:
            for g in buckets:
                rng.shuffle(buckets[g])

            folds = [[] for _ in range(k)]
            # Round-robin assign each group's units into folds:
            for g, members in buckets.items():
                for i, u in enumerate(members):
                    folds[i % k].append(u)

            # Optional: shuffle within fold for variety:
            for f in folds:
                rng.shuffle(f)
            return folds

        for r in range(n_repeats):
            if stratify and UNIT_GROUP is not None:
                folds = _kfold_indices_stratified(unit_ids_all, n_folds, UNIT_GROUP)
            else:
                folds = _kfold_indices(unit_ids_all, n_folds)

            for f in range(n_folds):
                test_ids = folds[f]
                train_ids = [u for i in range(n_folds) if i != f for u in folds[i]]

                _check_fold_minimums(train_ids, label=f"CV repeat={r} fold={f} split=TRAIN")
                _check_fold_minimums(test_ids,  label=f"CV repeat={r} fold={f} split=TEST")

                CV_SPLITS.append({
                    "repeat": int(r),
                    "fold": int(f),
                    "train_unit_ids": train_ids,
                    "test_unit_ids": test_ids,
                    "n_folds": int(n_folds),
                    "stratified": bool(stratify)})

        print(f"[CROSS-EVAL SPLIT] Built {len(CV_SPLITS)} CV split(s) ({n_repeats} repeat(s) x {n_folds} folds).")
    else:
        print("\n[CROSS-EVAL SPLIT] Generalization testing disabled (generalization.enabled=False).")

    # Print summary:
    print("\n[CROSS-EVAL SPLIT] Split objects now available:")
    print(f"   STABILITY_SPLITS: {len(STABILITY_SPLITS)}")
    print(f"   CV_SPLITS       : {len(CV_SPLITS)}")

In [ ]:
# __________________________________________________________________________________________________________
### BUILD FAST SPLIT -> (X, lengths) MATERIALIZATION HELPERS
#
# Uses:
#   - X_all, sequences_all_df, UNITS_DF
#
# Creates:
#   - SEQ_LOOKUP : dict[int] -> dict(row_start,row_end,length,subject_ID,session_ID)
#   - UNIT_ID_TO_SEQ_INDICES : dict[str] -> list[int]   (canonical unit->sequence mapping)
#   - get_sequence_indices_for_unit_ids(unit_ids) -> list[int]
#   - build_X_lengths_from_sequence_indices(seq_indices) -> (X, lengths, sequences_df)
#   - build_X_lengths_from_unit_ids(unit_ids) -> (X, lengths, sequences_df)
# __________________________________________________________________________________________________________

if not CROSS_MODEL_EVAL:
    print("[CROSS-EVAL MATERIALIZE] CROSS_MODEL_EVAL=False; skipping helper construction.")
else:
    # Build a lookup table for sequence slices (sequence_index --> row span):
    if "sequence_index" not in sequences_all_df.columns:
        # Earlier reconstruction may not create a sequence_index column; add a stable one here.
        sequences_all_df = sequences_all_df.reset_index(drop=True).copy()
        sequences_all_df["sequence_index"] = sequences_all_df.index.astype(int)

    SEQ_LOOKUP = {}
    for _, r in sequences_all_df.iterrows():
        seq_idx = int(r["sequence_index"])
        SEQ_LOOKUP[seq_idx] = {
            "row_start": int(r["row_start"]),
            "row_end": int(r["row_end"]),
            "length": int(r["length"]),
            "subject_ID": str(r.get("subject_ID", "")),
            "session_ID": str(r.get("session_ID", ""))}

    # ---------------------------------------
    # Unit --> list[int] sequence indices
    #      (Enforce that 'UNITS_DF["sequence_indices"]' == a real list of INTs for each row)
    # ---------------------------------------
    def _coerce_seq_indices(val):
        """
        Robustly coerce various representations into list[int].
        Accepts:
        - list/tuple/np.ndarray of ints
        - string like "[1, 2]" (JSON-ish) or "1,2" or "[1,2]"
        - scalar int-like
        Returns list[int].
        """
        if val is None:
            return []

        # If already list-like:
        if isinstance(val, (list, tuple, np.ndarray)):
            out = []
            for x in val:
                if x is None:
                    continue
                out.append(int(x))
            return out

        # Check string forms:
        if isinstance(val, str):
            s = val.strip()
            if s == "":
                return []
            # Try JSON list:
            if s.startswith("[") and s.endswith("]"):
                try:
                    parsed = json.loads(s.replace("'", '"'))
                    if isinstance(parsed, list):
                        return [int(x) for x in parsed if x is not None]
                except Exception:
                    # Fall through:
                    pass
                # Fallback == simply parse inside brackets:
                inner = s.strip("[]").strip()
                if inner == "":
                    return []
                return [int(x.strip()) for x in inner.split(",") if x.strip() != ""]
            # If comma-separated without brackets:
            if "," in s:
                return [int(x.strip()) for x in s.split(",") if x.strip() != ""]
            # If single INT-like string:
            try:
                return [int(s)]
            except Exception:
                return []

        # Scalar fallback:
        try:
            return [int(val)]
        except Exception:
            return []

    # Canonical dict: unit_id (str) --> list[int seq_index]:
    UNIT_ID_TO_SEQ_INDICES = {}
    for _, r in UNITS_DF.iterrows():
        unit_id = str(r["unit_id"])
        seqs = _coerce_seq_indices(r.get("sequence_indices", []))
        UNIT_ID_TO_SEQ_INDICES[unit_id] = seqs
    
    # Backwards-compatible alias (if any earlier cells used this name):
    UNIT_TO_SEQINDICES = UNIT_ID_TO_SEQ_INDICES

    # Optional: ensure 'UNITS_DF' column is actual list object (not strings):
    #     --> This makes downstream inspection/processing consistent
    UNITS_DF = UNITS_DF.copy()
    UNITS_DF["sequence_indices"] = UNITS_DF["unit_id"].astype(str).map(UNIT_ID_TO_SEQ_INDICES)

    # Helper function: unit_ids -> sequence indices (w/ stable ordering):
    def get_sequence_indices_for_unit_ids(unit_ids):
        seqs = []
        for u in unit_ids:
            seqs.extend(UNIT_ID_TO_SEQ_INDICES.get(str(u), []))

        # Keep only sequences that exist, then stable-unique-sort:
        seqs = [int(s) for s in seqs if int(s) in SEQ_LOOKUP]
        seqs = sorted(set(seqs))
        return seqs

    # Build (X, lengths, sequences_df) from a list of sequence indices:
    def build_X_lengths_from_sequence_indices(seq_indices):
        if seq_indices is None or len(seq_indices) == 0:
            X = np.empty((0, X_all.shape[1]), dtype=float)
            lengths = []
            seq_df = pd.DataFrame(columns=["sequence_index", "subject_ID", "session_ID", "length", "row_start", "row_end"])
            return X, lengths, seq_df

        chunks = []
        rows = []
        for seq_idx in seq_indices:
            info = SEQ_LOOKUP[int(seq_idx)]
            rs, re = info["row_start"], info["row_end"]
            chunk = X_all[rs:re + 1, :]
            chunks.append(chunk)
            rows.append({
                "sequence_index": int(seq_idx),
                "subject_ID": info["subject_ID"],
                "session_ID": info["session_ID"],
                "length": int(info["length"]),
                "row_start": int(rs),
                "row_end": int(re)})

        X = np.vstack(chunks) if len(chunks) else np.empty((0, X_all.shape[1]), dtype=float)
        seq_df = pd.DataFrame(rows)
        lengths = seq_df["length"].astype(int).tolist()

        # Sanity: lengths must sum to X rows
        if int(np.sum(lengths)) != int(X.shape[0]):
            handle_error(
                "[CROSS-EVAL MATERIALIZE] Sum(lengths) != X rows. "
                f"sum(lengths)={int(np.sum(lengths))}, X.shape[0]={int(X.shape[0])}")

        return X, lengths, seq_df

    # Convenience wrapper: unit_ids -> (X, lengths, seq_df):
    def build_X_lengths_from_unit_ids(unit_ids):
        seq_indices = get_sequence_indices_for_unit_ids(unit_ids)
        return build_X_lengths_from_sequence_indices(seq_indices)

    print("\n[CROSS-EVAL MATERIALIZE] Helpers ready.")
    print(f"   # sequences in lookup = {len(SEQ_LOOKUP)}")
    print(f"   # units in mapping    = {len(UNIT_ID_TO_SEQ_INDICES)}")

----------

### **All training data & models should be loaded in now; next we proceed to actual evaluation:**

**First,** we check state distributions in the training data to make sure all states are used, etc.:

In [ ]:
# __________________________________________________________________________________________________________
### DECODING QC GATE (pre-filter candidate models before stability/generalization)
#
# Goal:
#   - Quickly detect collapsed/degenerate models *before* running stability/generalization.
#   - Optionally DROP failed protocols from SCANNED_PROTOCOLS and models_index_df so downstream
#     stages only see eligible candidates.
#
# Uses:
#   - SCANNED_PROTOCOLS, models_index_df
#   - X_all, lengths_all, SEQ_LOOKUP (from reconstruction/materialization)
#   - DATASET_MODELS_DIR (from scanner cell)
#
# Creates:
#   - DECODING_QC_SUMMARY_DF
#   - (optional) DECODING_QC_DETAILS_DF  (per-sequence; off by default)
#   - FILTERED_PROTOCOLS (list[str]) and updates:
#       * SCANNED_PROTOCOLS
#       * models_index_df
#
# Side effects:
#   - Writes decoding_qc_summary.csv (and optionally decoding_qc_details.csv) under:
#       DATASET_MODELS_DIR / "_cross_model_evaluation"
# __________________________________________________________________________________________________________

if not CROSS_MODEL_EVAL:
    print("[DECODING QC] CROSS_MODEL_EVAL=False; skipping decoding QC gate.")
    DECODING_QC_SUMMARY_DF = None
    DECODING_QC_DETAILS_DF = None
    FILTERED_PROTOCOLS = []
else:
    # Sanity-checks:
    if "SCANNED_PROTOCOLS" not in globals() or not SCANNED_PROTOCOLS:
        raise RuntimeError("[DECODING QC ERROR] SCANNED_PROTOCOLS missing/empty. Run scanner cell first.")
    if "models_index_df" not in globals() or models_index_df is None or models_index_df.empty:
        raise RuntimeError("[DECODING QC ERROR] models_index_df missing/empty. Run scanner cell first.")
    if "SEQ_LOOKUP" not in globals() or not SEQ_LOOKUP:
        raise RuntimeError("[DECODING QC ERROR] SEQ_LOOKUP missing/empty. Run reconstruction/materialization first.")
    if "X_all" not in globals() or X_all is None or X_all.shape[0] == 0:
        raise RuntimeError("[DECODING QC ERROR] X_all missing/empty. Run reconstruction cell first.")
    if "lengths_all" not in globals() or lengths_all is None or len(lengths_all) == 0:
        raise RuntimeError("[DECODING QC ERROR] lengths_all missing/empty. Run reconstruction cell first.")

    # Set output directory:
    CROSS_EVAL_DIR = Path(DATASET_MODELS_DIR) / "_cross_model_evaluation"
    CROSS_EVAL_DIR.mkdir(parents=True, exist_ok=True)

    # QC parameters (defaults; can be wired into YAML later)
    QC_CFG = dict(config.get("cross_model_evaluation", {}).get("decoding_qc", {}))

    # If no downstream cross-evaluation stages are enabled, decoding QC should not block manual selection:
    STAB_ENABLED = bool(config.get("cross_model_evaluation", {}).get("stability", {}).get("enabled", True))
    GEN_ENABLED  = bool(config.get("cross_model_evaluation", {}).get("generalization", {}).get("enabled", True))
    ANY_CROSS_EVAL_STAGE_ENABLED = (STAB_ENABLED or GEN_ENABLED)

    # Set minimum occupancy (fraction of all timepoints) for a state to count as "used":
    MIN_STATE_OCC_FRACTION = float(QC_CFG.get("min_state_occupancy_fraction", 0.01))  # 1%

    # Collapse if too few states meet minimum occupancy:
    MIN_USED_STATES_ABS = QC_CFG.get("min_used_states_abs", None)      # optional hard number
    MIN_USED_STATES_FRAC = float(QC_CFG.get("min_used_states_frac", 0.5))  # at least 50% of K used

    # Dominance check: if a single state takes > DOMINANT_OCC_THRESHOLD of a sequence:
    DOMINANT_OCC_THRESHOLD = float(QC_CFG.get("dominant_occ_threshold", 0.90))
    MAX_DOMINANT_SEQ_FRACTION = float(QC_CFG.get("max_dominant_sequence_fraction", 0.25))  # allow up to 25% sequences dominated

    # Transition degeneracy flags (soft by default):
    MAX_MEAN_STICKINESS = float(QC_CFG.get("max_mean_stickiness", 0.98))  # trace(T)/K
    MIN_MEAN_STICKINESS = float(QC_CFG.get("min_mean_stickiness", 0.02))

    # Posterior certainty (optional; may be expensive on huge datasets):
    COMPUTE_POSTERIOR = bool(QC_CFG.get("compute_posterior", False))
    MIN_MEAN_MAX_POSTERIOR = float(QC_CFG.get("min_mean_max_posterior", 0.55))  # soft flag unless you want it hard

    # Behavior: drop failed models from candidate set:
    DROP_FAILED_MODELS = bool(QC_CFG.get("drop_failed_models", True))

    if not ANY_CROSS_EVAL_STAGE_ENABLED:
        print("[DECODING QC] stability.enabled=False and generalization.enabled=False; "
              "QC gate will run for reporting only (no candidate dropping / hard-stop).")
        DROP_FAILED_MODELS = False  # <-- minimal local override to avoid blocking manual selection

    print("\n" + "=" * 104)
    print("[DECODING QC] Running decoding QC gate (pre-filter candidates)")
    print("=" * 104)
    print("[DECODING QC] Parameters:")
    print(f"   min_state_occupancy_fraction   = {MIN_STATE_OCC_FRACTION}")
    print(f"   min_used_states_frac           = {MIN_USED_STATES_FRAC}" + (f" (min_used_states_abs={MIN_USED_STATES_ABS})" if MIN_USED_STATES_ABS is not None else ""))
    print(f"   dominant_occ_threshold         = {DOMINANT_OCC_THRESHOLD}")
    print(f"   max_dominant_sequence_fraction = {MAX_DOMINANT_SEQ_FRACTION}")
    print(f"   compute_posterior              = {COMPUTE_POSTERIOR}")
    print(f"   drop_failed_models             = {DROP_FAILED_MODELS}")

    # Helper: apply PCA (if present for protocol):
    def _apply_protocol_pca_if_available(proto_info: dict, X: np.ndarray) -> tuple:
        """
        Returns (X_eff, pca_applied_str).
        Preference order:
          1) pca_refit_on_X_all.joblib
          2) pca_fit_on_X_train.joblib
          3) None (no PCA)
        """
        pca_all = proto_info.get("pca_refit_on_X_all_path", None)
        pca_train = proto_info.get("pca_fit_on_X_train_path", None)

        if pca_all and Path(pca_all).exists():
            pca_obj = joblib.load(pca_all)
            X_eff = pca_obj.transform(X)
            return X_eff, f"PCA(refit on X_all) {X.shape[1]} -> {X_eff.shape[1]}"
        if pca_train and Path(pca_train).exists():
            pca_obj = joblib.load(pca_train)
            X_eff = pca_obj.transform(X)
            return X_eff, f"PCA(fit on X_train) {X.shape[1]} -> {X_eff.shape[1]}"
        return X, "PCA not used"

    # Helper function -- compute per-sequence occupancies + dominance from a global state vector:
    def _per_sequence_stats_from_states(states: np.ndarray, K: int) -> pd.DataFrame:
        rows = []
        for seq_idx, info in SEQ_LOOKUP.items():
            rs, re = int(info["row_start"]), int(info["row_end"])
            st = states[rs:re + 1]
            if st.size == 0:
                continue
            counts = np.bincount(st.astype(int), minlength=K).astype(float)
            occ = counts / float(st.size)
            rows.append({
                "sequence_index": int(seq_idx),
                "subject_ID": info.get("subject_ID", ""),
                "session_ID": info.get("session_ID", ""),
                "length": int(info.get("length", st.size)),
                "dominant_occ": float(np.max(occ)) if occ.size else np.nan,
                "dominant_state": int(np.argmax(occ)) if occ.size else None})
        return pd.DataFrame(rows)

    # Helper function -- compute empirical transition matrix from Viterbi states (per-sequence boundaries respected):
    def _empirical_transition_matrix(states: np.ndarray, K: int) -> np.ndarray:
        T = np.zeros((K, K), dtype=float)
        for seq_idx, info in SEQ_LOOKUP.items():
            rs, re = int(info["row_start"]), int(info["row_end"])
            st = states[rs:re + 1].astype(int)
            if st.size < 2:
                continue
            for a, b in zip(st[:-1], st[1:]):
                if 0 <= a < K and 0 <= b < K:
                    T[a, b] += 1.0
        # Row-normalize where possible:
        row_sums = T.sum(axis=1, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            Tn = np.divide(T, row_sums, where=(row_sums > 0))
        # For rows with 0 outgoing (never visited), leave zeros:
        return Tn

    # ----------------------------
    # Main loop: QC for each protocol model:
    # ----------------------------
    summary_rows = []
    details_frames = []

    eligible_protocols = []
    failed_protocols = []

    for proto_name, proto_info in SCANNED_PROTOCOLS.items():
        final_model_path = Path(proto_info.get("final_model_path", ""))
        if not final_model_path.exists():
            handle_error(f"[DECODING QC WARN] Protocol '{proto_name}': final_model_path missing: {final_model_path}. Skipping.")
            failed_protocols.append(proto_name)
            continue

        # Load model:
        try:
            model = joblib.load(final_model_path)
        except Exception as exc:
            handle_error(f"[DECODING QC WARN] Protocol '{proto_name}': failed to load model: {exc}. Skipping.")
            failed_protocols.append(proto_name)
            continue

        # Infer K:
        K = getattr(model, "n_components", None)
        if K is None:
            handle_error(f"[DECODING QC WARN] Protocol '{proto_name}': model has no n_components; cannot QC. Skipping.")
            failed_protocols.append(proto_name)
            continue
        K = int(K)

        # Apply PCA (if needed):
        X_eff, pca_msg = _apply_protocol_pca_if_available(proto_info, X_all)
        print("\n" + "-" * 104)
        print(f"[DECODING QC] Protocol '{proto_name}' | K={K} | {pca_msg}")

        # Decode states (Viterbi):
        try:
            states = model.predict(X_eff, lengths_all).astype(int)
        except Exception as exc:
            handle_error(f"[DECODING QC WARN] Protocol '{proto_name}': model.predict failed: {exc}. Skipping.")
            failed_protocols.append(proto_name)
            continue

        if states.shape[0] != X_eff.shape[0]:
            handle_error(
                f"[DECODING QC WARN] Protocol '{proto_name}': decoded states length mismatch "
                f"(states={states.shape[0]}, X={X_eff.shape[0]}). Skipping.")
            failed_protocols.append(proto_name)
            continue

        # Global occupancy:
        counts = np.bincount(states, minlength=K).astype(float)
        occ = counts / float(states.size)
        n_used_states = int(np.sum(occ >= MIN_STATE_OCC_FRACTION))
        frac_used_states = float(n_used_states) / float(K) if K > 0 else 0.0

        # Per-sequence dominance:
        seq_stats = _per_sequence_stats_from_states(states, K)
        if seq_stats.empty:
            handle_error(f"[DECODING QC WARN] Protocol '{proto_name}': no per-sequence stats computed. Skipping.")
            failed_protocols.append(proto_name)
            continue

        dominant_frac = float(np.mean(seq_stats["dominant_occ"].astype(float) >= DOMINANT_OCC_THRESHOLD))

        # Calculate transition "stickiness":
        T_emp = _empirical_transition_matrix(states, K)
        diag_mass = float(np.trace(T_emp))
        mean_stickiness = float(diag_mass / float(K)) if K > 0 else np.nan

        # Calculate posterior certainty (optional):
        mean_max_post = np.nan
        if COMPUTE_POSTERIOR:
            try:
                post = model.predict_proba(X_eff, lengths_all)  # (n_obs, K)
                mean_max_post = float(np.mean(np.max(post, axis=1)))
            except Exception as exc:
                print(f"[DECODING QC WARN] Protocol '{proto_name}': predict_proba failed ({exc}); leaving posterior metrics as NaN.")

        ### Pass/fail logic checks:
        fail_reasons = []

        # Used-states gate:
        if MIN_USED_STATES_ABS is not None:
            if n_used_states < int(MIN_USED_STATES_ABS):
                fail_reasons.append(f"too_few_states_used: used={n_used_states} < min_abs={int(MIN_USED_STATES_ABS)}")
        else:
            if frac_used_states < MIN_USED_STATES_FRAC:
                fail_reasons.append(f"too_few_states_used: frac_used={frac_used_states:.3f} < min_frac={MIN_USED_STATES_FRAC:.3f}")

        # Dominance gate:
        if dominant_frac > MAX_DOMINANT_SEQ_FRACTION:
            fail_reasons.append(
                f"too_many_dominant_sequences: frac={dominant_frac:.3f} > max={MAX_DOMINANT_SEQ_FRACTION:.3f} "
                f"(threshold={DOMINANT_OCC_THRESHOLD:.2f})")

        # Stickiness soft-flag (do not hard-fail by default):
        stickiness_flag = (mean_stickiness > MAX_MEAN_STICKINESS) or (mean_stickiness < MIN_MEAN_STICKINESS)

        # Posterior soft-flag:
        posterior_flag = (COMPUTE_POSTERIOR and (not np.isnan(mean_max_post)) and (mean_max_post < MIN_MEAN_MAX_POSTERIOR))

        passed = (len(fail_reasons) == 0)

        print(f"[DECODING QC] used_states = {n_used_states}/{K} (frac={frac_used_states:.3f}; min_occ={MIN_STATE_OCC_FRACTION})")
        print(f"[DECODING QC] dominant_seq_fraction = {dominant_frac:.3f} (dominant if >= {DOMINANT_OCC_THRESHOLD:.2f})")
        print(f"[DECODING QC] mean_stickiness = {mean_stickiness:.3f}" + (" [FLAG]" if stickiness_flag else ""))
        if COMPUTE_POSTERIOR:
            print(f"[DECODING QC] mean_max_posterior = {mean_max_post:.3f}" + (" [FLAG]" if posterior_flag else ""))

        if passed:
            print("[DECODING QC] RESULT: PASS")
            eligible_protocols.append(proto_name)
        else:
            print("[DECODING QC] RESULT: FAIL")
            for r in fail_reasons:
                print(f"   - {r}")
            failed_protocols.append(proto_name)

        summary_rows.append({
            "protocol": proto_name,
            "K": K,
            "final_model_path": str(final_model_path),
            "pca_note": pca_msg,
            "min_state_occupancy_fraction": MIN_STATE_OCC_FRACTION,
            "n_used_states": n_used_states,
            "frac_used_states": frac_used_states,
            "dominant_occ_threshold": DOMINANT_OCC_THRESHOLD,
            "dominant_sequence_fraction": dominant_frac,
            "mean_stickiness": mean_stickiness,
            "stickiness_flag": bool(stickiness_flag),
            "compute_posterior": bool(COMPUTE_POSTERIOR),
            "mean_max_posterior": mean_max_post,
            "posterior_flag": bool(posterior_flag),
            "passed_decoding_qc": bool(passed),
            "fail_reasons": "; ".join(fail_reasons) if fail_reasons else ""})

        # Optional per-sequence details:
        if bool(QC_CFG.get("save_per_sequence_details", False)):
            tmp = seq_stats.copy()
            tmp["protocol"] = proto_name
            tmp["K"] = K
            details_frames.append(tmp)

    # ---------------------------------
    # Summaries & optional filtering:
    # ---------------------------------
    DECODING_QC_SUMMARY_DF = pd.DataFrame(summary_rows).sort_values(["passed_decoding_qc", "protocol"], ascending=[False, True]).reset_index(drop=True)

    print("\n[DECODING QC] Summary:")
    display(DECODING_QC_SUMMARY_DF)

    summary_path = CROSS_EVAL_DIR / "decoding_qc_summary.csv"
    DECODING_QC_SUMMARY_DF.to_csv(summary_path, index=False)
    print(f"[DECODING QC] Wrote: {summary_path}")

    DECODING_QC_DETAILS_DF = None
    if details_frames:
        DECODING_QC_DETAILS_DF = pd.concat(details_frames, ignore_index=True)
        details_path = CROSS_EVAL_DIR / "decoding_qc_details_per_sequence.csv"
        DECODING_QC_DETAILS_DF.to_csv(details_path, index=False)
        print(f"[DECODING QC] Wrote: {details_path}")

    # Drop failed protocols from candidates (optional):
    FILTERED_PROTOCOLS = eligible_protocols

    if DROP_FAILED_MODELS:
        before_n = len(SCANNED_PROTOCOLS)
        SCANNED_PROTOCOLS = {p: SCANNED_PROTOCOLS[p] for p in FILTERED_PROTOCOLS if p in SCANNED_PROTOCOLS}
        after_n = len(SCANNED_PROTOCOLS)

        # Also filter 'models_index_df':
        models_index_df = models_index_df[models_index_df["protocol"].isin(FILTERED_PROTOCOLS)].copy()
        models_index_df = models_index_df.sort_values(["protocol"]).reset_index(drop=True)

        print("\n[DECODING QC] Candidate set filtered based on decoding QC:")
        print(f"   protocols before = {before_n}")
        print(f"   protocols after  = {after_n}")
        if failed_protocols:
            print(f"   failed protocols = {failed_protocols}")

        if after_n == 0:
            if ANY_CROSS_EVAL_STAGE_ENABLED:
                handle_error(
                    "[DECODING QC ERROR] All protocols failed decoding QC; cannot proceed to stability/generalization. "
                    "Review decoding_qc_summary.csv for details.")
            else:
                print("[DECODING QC WARN] All protocols failed decoding QC, but stability/generalization are disabled; "
                      "continuing to allow manual model selection.")

        print("\n[DECODING QC] Updated models_index_df (eligible only):")
        display(models_index_df)
    else:
        print("\n[DECODING QC] DROP_FAILED_MODELS=False; leaving candidate set unchanged.")
        print(f"   eligible protocols = {eligible_protocols}")
        print(f"   failed protocols   = {failed_protocols}")

**Second,** we perform model stability tests (then generalization):

In [ ]:
# __________________________________________________________________________________________________________
### STABILITY EVALUATION (per protocol; repeated half-splits)
#
# Creates:
#   - STABILITY_RESULTS_LONG_DF   : repeat-level results (optional)
#   - STABILITY_SUMMARY_DF        : per-protocol summary table
#
# Notes:
#   - mode == 'fixed_model': uses each protocol's final model (and PCA refit on X_all if present)
#   - mode == 'refit_models': not implemented yet (we can add next)
# __________________________________________________________________________________________________________

if (not CROSS_MODEL_EVAL) or (not stab_enabled):
    print("[STABILITY] Stability evaluation skipped "
          f"([CONFIG:] CROSS_MODEL_EVAL={CROSS_MODEL_EVAL}, stability.enabled={stab_enabled}).")
    STABILITY_RESULTS_LONG_DF = pd.DataFrame()
    STABILITY_SUMMARY_DF = pd.DataFrame()
else:
    # Read YAML toggles:
    STAB_ENABLED = bool(STABILITY_CFG.get("enabled", True))
    STAB_MODE = str(STABILITY_CFG.get("mode", "fixed_model")).strip().lower()
    STAB_STRATEGY = str(STABILITY_CFG.get("split_strategy", "blocked_random")).strip()
    STAB_N_REPEATS = int(STABILITY_CFG.get("n_repeats", 20))
    STAB_BLOCK_LEN = int(STABILITY_CFG.get("block_len", 2))

    STAB_METRICS = dict(STABILITY_CFG.get("metrics", {}))
    DO_OCC_CORR = bool(STAB_METRICS.get("occupancy_corr", True))
    DO_TRANS_FROB = bool(STAB_METRICS.get("transition_frobenius", True))
    DO_DWELL_JSD = bool(STAB_METRICS.get("dwell_jsd", True))

    STAB_REPORT = dict(STABILITY_CFG.get("report", {}))
    SAVE_REPEAT_DETAILS = bool(STAB_REPORT.get("save_per_repeat_details", False))

    # Minimums (already enforced upstream in split construction, but kept here for safety):
    MIN_SEQ_LEN = int(MINIMUMS_CFG.get("min_sequence_length", 1))

    if not STAB_ENABLED:
        print("[STABILITY] stability.enabled=False -> skipping stability evaluation.")
        STABILITY_RESULTS_LONG_DF = pd.DataFrame()
        STABILITY_SUMMARY_DF = pd.DataFrame()
    else:
        if STAB_MODE not in ("fixed_model", "refit_models"):
            raise RuntimeError(f"[STABILITY ERROR] Unknown stability.mode='{STAB_MODE}'. Expected 'fixed_model' or 'refit_models'.")

        if STAB_MODE == "refit_models":
            raise RuntimeError(
                "[STABILITY] stability.mode='refit_models' is not implemented yet in this script. "
                "If you want it now, we can implement it next (it requires per-split model fitting + Hungarian alignment).")

        if "STABILITY_SPLITS" not in globals() or not STABILITY_SPLITS:
            raise RuntimeError("[STABILITY ERROR] STABILITY_SPLITS is missing/empty. Run the split-building cell first.")

        # ----------------------------
        # Helper functions:
        # ----------------------------
        # Map unit_id -> list[int] sequence_indices:
        _unit_to_seq = {}
        for _, r in UNITS_DF.iterrows():
            unit_id = str(r["unit_id"])
            seq_idx = r["sequence_indices"]
            if isinstance(seq_idx, str):
                # Tolerate stringified lists, if ever present:
                try:
                    seq_idx = json.loads(seq_idx)
                except Exception:
                    # Last resort == strip any brackets & split:
                    s = seq_idx.strip().strip("[]")
                    seq_idx = [int(x.strip()) for x in s.split(",") if x.strip() != ""]
            if isinstance(seq_idx, (list, tuple, np.ndarray)):
                _unit_to_seq[unit_id] = [int(x) for x in list(seq_idx)]
            else:
                _unit_to_seq[unit_id] = [int(seq_idx)]

        def _seq_indices_for_units(unit_ids) -> list:
            out = []
            for u in unit_ids:
                u = str(u)
                if u not in _unit_to_seq:
                    handle_error(f"[STABILITY WARN] unit_id '{u}' not found in UNITS_DF mapping; skipping.")
                    continue
                out.extend(_unit_to_seq[u])
            # Keep stable ordering & remove duplicates:
            seen = set()
            dedup = []
            for x in out:
                if x not in seen:
                    seen.add(x)
                    dedup.append(int(x))
            return dedup

        def _split_states_by_lengths(z_concat: np.ndarray, lengths: list) -> list:
            """Convert concatenated z (len = sum(lengths)) into list of per-sequence arrays."""
            out = []
            cursor = 0
            for L in lengths:
                L = int(L)
                out.append(z_concat[cursor:cursor+L])
                cursor += L
            return out

        def _compute_occupancy(z_list: list, K: int) -> np.ndarray:
            """Occupancy vector: fraction of timepoints in each state across all sequences."""
            counts = np.zeros((K,), dtype=float)
            total = 0
            for z in z_list:
                if z is None or len(z) == 0:
                    continue
                total += int(len(z))
                c = np.bincount(z.astype(int), minlength=K).astype(float)
                counts += c
            if total <= 0:
                return np.full((K,), np.nan)
            return counts / float(total)

        def _compute_transition_matrix(z_list: list, K: int) -> np.ndarray:
            """
            Transition probability matrix estimated from state sequences.
            Row-normalized counts; rows with zero outgoing transitions become zeros.
            """
            T = np.zeros((K, K), dtype=float)
            for z in z_list:
                if z is None or len(z) < 2:
                    continue
                z = z.astype(int)
                for a, b in zip(z[:-1], z[1:]):
                    if 0 <= a < K and 0 <= b < K:
                        T[a, b] += 1.0
            row_sums = T.sum(axis=1, keepdims=True)
            with np.errstate(divide="ignore", invalid="ignore"):
                Tn = np.divide(T, row_sums, out=np.zeros_like(T), where=(row_sums > 0))
            return Tn

        def _dwell_lengths_by_state(z_list: list, K: int) -> list:
            """Return list of length-K, each entry is a list of dwell lengths for that state."""
            dwells = [[] for _ in range(K)]
            for z in z_list:
                if z is None or len(z) == 0:
                    continue
                z = z.astype(int)
                run_state = int(z[0])
                run_len = 1
                for s in z[1:]:
                    s = int(s)
                    if s == run_state:
                        run_len += 1
                    else:
                        if 0 <= run_state < K:
                            dwells[run_state].append(int(run_len))
                        run_state = s
                        run_len = 1
                # last run
                if 0 <= run_state < K:
                    dwells[run_state].append(int(run_len))
            return dwells

        def _pmf_from_lengths(lengths: list, max_len: int, eps: float = 1e-12) -> np.ndarray:
            """Convert a list of positive integer lengths into a PMF over 1..max_len."""
            if max_len <= 0:
                return np.array([1.0], dtype=float)
            counts = np.zeros((max_len,), dtype=float)
            for L in lengths:
                L = int(L)
                if L <= 0:
                    continue
                if L > max_len:
                    L = max_len
                counts[L - 1] += 1.0
            counts = counts + eps
            return counts / counts.sum()

        def _js_divergence(p: np.ndarray, q: np.ndarray, eps: float = 1e-12) -> float:
            """Jensen-Shannon divergence (natural log)."""
            p = np.asarray(p, dtype=float)
            q = np.asarray(q, dtype=float)
            p = p / (p.sum() + eps)
            q = q / (q.sum() + eps)
            m = 0.5 * (p + q)

            def _kl(a, b):
                a = np.clip(a, eps, None)
                b = np.clip(b, eps, None)
                return float(np.sum(a * np.log(a / b)))

            return 0.5 * _kl(p, m) + 0.5 * _kl(q, m)

        def _mean_dwell_jsd(zA: list, zB: list, K: int) -> float:
            """
            Mean JSD across states of dwell-length distributions.
            Uses a shared max_len per state = max observed in A*B (clipped at that).
            """
            dwA = _dwell_lengths_by_state(zA, K)
            dwB = _dwell_lengths_by_state(zB, K)
            jsds = []
            for k in range(K):
                max_len = 0
                if dwA[k]:
                    max_len = max(max_len, max(dwA[k]))
                if dwB[k]:
                    max_len = max(max_len, max(dwB[k]))
                if max_len <= 0:
                    continue
                p = _pmf_from_lengths(dwA[k], max_len=max_len)
                q = _pmf_from_lengths(dwB[k], max_len=max_len)
                jsds.append(_js_divergence(p, q))
            if not jsds:
                return float("nan")
            return float(np.mean(jsds))

        # ------------------------------------------------
        # Evaluate each protocol in 'fixed_model' mode:
        # ------------------------------------------------
        stability_rows = []
        summary_rows = []

        print("\n" + "=" * 88)
        print("[STABILITY] Running stability evaluation (fixed_model mode)")
        print("=" * 88)
        print(f"[STABILITY] strategy={STAB_STRATEGY}, n_repeats={len(STABILITY_SPLITS)} (requested={STAB_N_REPEATS}), block_len={STAB_BLOCK_LEN}")
        print(f"[STABILITY] metrics: occupancy_corr={DO_OCC_CORR}, transition_frobenius={DO_TRANS_FROB}, dwell_jsd={DO_DWELL_JSD}")

        for proto_name, info in SCANNED_PROTOCOLS.items():
            payload = info.get("chosen_payload", {}) if isinstance(info.get("chosen_payload", None), dict) else {}

            final_model_path = Path(info["final_model_path"])
            if not final_model_path.exists():
                handle_error(f"[STABILITY WARN] Protocol '{proto_name}': final model not found: {final_model_path}. Skipping.")
                continue

            # Load final model (was trained on 'X_all' in prior script, assuming 'REFIT_ON_ALL_DATA'=True):
            model = joblib.load(final_model_path)
            K = int(getattr(model, "n_components", payload.get("chosen_K", None) or 0))
            if K <= 0:
                handle_error(f"[STABILITY WARN] Protocol '{proto_name}': could not infer K from model. Skipping.")
                continue

            # If a PCA refit on 'X_all' exists for this protocol, apply it before scoring/predicting:
            X_eff = X_all
            pca_all_path = info.get("pca_refit_on_X_all_path", None)
            if pca_all_path is not None:
                pca_all_path = Path(pca_all_path)
                if pca_all_path.exists():
                    pca_obj = joblib.load(pca_all_path)
                    X_eff = pca_obj.transform(X_all)
                    print(f"\n[STABILITY] Protocol '{proto_name}': PCA(refit on X_all) applied: {X_all.shape[1]} -> {X_eff.shape[1]}")
                else:
                    handle_error(f"[STABILITY WARN] Protocol '{proto_name}': pca_refit_on_X_all_path missing on disk: {pca_all_path}. Proceeding without PCA.")

            # Predict Viterbi states once for ALL sequences -- then subset per split:
            try:
                z_concat = model.predict(X_eff, lengths_all)
                z_by_seq = _split_states_by_lengths(np.asarray(z_concat, dtype=int), lengths_all)
            except Exception as exc:
                handle_error(f"[STABILITY WARN] Protocol '{proto_name}': model.predict failed: {exc}. Skipping.")
                continue

            # Quick sanity: 'z_by_seq' count should match number of sequences:
            if len(z_by_seq) != len(lengths_all):
                handle_error(
                    f"[STABILITY WARN] Protocol '{proto_name}': mismatch #sequences in z_by_seq ({len(z_by_seq)}) vs lengths_all ({len(lengths_all)}).")

            # Evaluate all stability splits:
            for rep_idx, split in enumerate(STABILITY_SPLITS):
                units_A = split.get("A_unit_ids", None)
                units_B = split.get("B_unit_ids", None)

                # Backward-compatible fallbacks (in case older split-builders were used):
                if units_A is None:
                    units_A = split.get("A_units", split.get("units_A", split.get("train_units", None)))
                if units_B is None:
                    units_B = split.get("B_units", split.get("units_B", split.get("test_units", None)))

                if units_A is None or units_B is None:
                    handle_error(
                        f"[STABILITY WARN] Split #{rep_idx}: missing unit lists. "
                        f"Expected keys 'A_unit_ids'/'B_unit_ids'. Keys present: {list(split.keys())}")
                    continue

                seq_A = _seq_indices_for_units(units_A)
                seq_B = _seq_indices_for_units(units_B)

                # Filter sequences by minimum length (purely for safety; this should already have been enforced upstream):
                seq_A = [i for i in seq_A if int(lengths_all[int(i)]) >= MIN_SEQ_LEN]
                seq_B = [i for i in seq_B if int(lengths_all[int(i)]) >= MIN_SEQ_LEN]

                zA = [z_by_seq[int(i)] for i in seq_A if 0 <= int(i) < len(z_by_seq)]
                zB = [z_by_seq[int(i)] for i in seq_B if 0 <= int(i) < len(z_by_seq)]

                # Compute requested metrics:
                occ_corr = np.nan
                trans_frob = np.nan
                dwell_jsd = np.nan

                if DO_OCC_CORR:
                    occA = _compute_occupancy(zA, K)
                    occB = _compute_occupancy(zB, K)
                    if np.any(np.isnan(occA)) or np.any(np.isnan(occB)):
                        occ_corr = np.nan
                    else:
                        # Calculate Pearson correlation of occupancy vectors:
                        if np.std(occA) == 0 or np.std(occB) == 0:
                            occ_corr = np.nan
                        else:
                            occ_corr = float(np.corrcoef(occA, occB)[0, 1])

                if DO_TRANS_FROB:
                    TA = _compute_transition_matrix(zA, K)
                    TB = _compute_transition_matrix(zB, K)
                    trans_frob = float(np.linalg.norm(TA - TB, ord="fro"))

                if DO_DWELL_JSD:
                    dwell_jsd = _mean_dwell_jsd(zA, zB, K)

                stability_rows.append({
                    "protocol": proto_name,
                    "K": int(K),
                    "repeat_index": int(rep_idx),
                    "n_units_A": int(len(units_A)),
                    "n_units_B": int(len(units_B)),
                    "n_sequences_A": int(len(seq_A)),
                    "n_sequences_B": int(len(seq_B)),
                    "n_obs_A": int(sum(int(lengths_all[int(i)]) for i in seq_A)) if seq_A else 0,
                    "n_obs_B": int(sum(int(lengths_all[int(i)]) for i in seq_B)) if seq_B else 0,
                    "occupancy_corr": occ_corr,
                    "transition_frobenius": trans_frob,
                    "dwell_jsd": dwell_jsd})

            # Build per-protocol summaries:
            dfp = pd.DataFrame([r for r in stability_rows if r["protocol"] == proto_name])
            if dfp.empty:
                handle_error(f"[STABILITY WARN] Protocol '{proto_name}': no stability rows computed.")
                continue

            summary_rows.append({
                "protocol": proto_name,
                "K": int(K),
                "stability_mode": STAB_MODE,
                "n_repeats": int(dfp.shape[0]),
                "occupancy_corr_mean": float(dfp["occupancy_corr"].mean(skipna=True)) if DO_OCC_CORR else np.nan,
                "occupancy_corr_std": float(dfp["occupancy_corr"].std(skipna=True)) if DO_OCC_CORR else np.nan,
                "transition_frobenius_mean": float(dfp["transition_frobenius"].mean(skipna=True)) if DO_TRANS_FROB else np.nan,
                "transition_frobenius_std": float(dfp["transition_frobenius"].std(skipna=True)) if DO_TRANS_FROB else np.nan,
                "dwell_jsd_mean": float(dfp["dwell_jsd"].mean(skipna=True)) if DO_DWELL_JSD else np.nan,
                "dwell_jsd_std": float(dfp["dwell_jsd"].std(skipna=True)) if DO_DWELL_JSD else np.nan,
                "final_model_path": str(final_model_path)})

            print(f"\n[STABILITY] Protocol '{proto_name}' complete.")
            if DO_OCC_CORR:
                print(f"   occupancy_corr: mean={summary_rows[-1]['occupancy_corr_mean']:.3f}, std={summary_rows[-1]['occupancy_corr_std']:.3f}")
            if DO_TRANS_FROB:
                print(f"   transition_frobenius: mean={summary_rows[-1]['transition_frobenius_mean']:.3f}, std={summary_rows[-1]['transition_frobenius_std']:.3f}")
            if DO_DWELL_JSD:
                print(f"   dwell_jsd: mean={summary_rows[-1]['dwell_jsd_mean']:.3f}, std={summary_rows[-1]['dwell_jsd_std']:.3f}")

        STABILITY_RESULTS_LONG_DF = pd.DataFrame(stability_rows)
        STABILITY_SUMMARY_DF = pd.DataFrame(summary_rows).sort_values(["protocol"]).reset_index(drop=True)

        print("\n[STABILITY] Per-protocol stability summary:")
        display(STABILITY_SUMMARY_DF)

        # Optional saving:
        CROSS_EVAL_DIR = Path(DATASET_MODELS_DIR) / "_cross_model_evaluation"
        CROSS_EVAL_DIR.mkdir(parents=True, exist_ok=True)

        summary_path = CROSS_EVAL_DIR / "stability_summary.csv"
        STABILITY_SUMMARY_DF.to_csv(summary_path, index=False)
        print(f"\n[STABILITY] Wrote: {summary_path}")

        if SAVE_REPEAT_DETAILS:
            long_path = CROSS_EVAL_DIR / "stability_repeat_level.csv"
            STABILITY_RESULTS_LONG_DF.to_csv(long_path, index=False)
            print(f"[STABILITY] Wrote: {long_path}")
        else:
            print("[STABILITY] save_per_repeat_details=False -> repeat-level CSV not written (in-memory DF is still available as STABILITY_RESULTS_LONG_DF).")

**Notes on the above metrics / outputs:**

- **Occupancy correlation**: Measures whether the fraction of time spent in each state is consistent across random half-splits (higher = better).
- **Transition Frobenius distance**: Measures how similar the state-to-state transition matrices are between splits (lower = better).
- **Dwell time Jensen–Shannon divergence**: Measures whether state duration distributions match across splits (lower = better).

**Overall**, the stability test is effectively answering “If I had drawn a slightly different sample of subjects, would I get the same brain states?”.

What generalization testing will add: Stability answers: “Is the structure reproducible?”; generalization will answer: “Does it explain unseen data better?”

-------
-------
**NOTE:** Add visual QC outputs either here, or later on:

- Histogram of occupancy correlations across repeats
- Boxplot of transition Frobenius distances
- Side-by-side heatmaps of average transition matrices

-------
-------

Next up, generalization tests:

In [ ]:
# __________________________________________________________________________________________________________
### GENERALIZATION (K-fold CV) EVALUATION (per protocol; refit within each fold)
#
# Goal:
#   - For each protocol selected in SCANNED_PROTOCOLS / models_index_df:
#       * Use that protocol's selected K (from chosen_model.json)
#       * Refit HMM within each CV fold (train folds only)
#       * Score on held-out fold (test units)
#       * Report logL per timepoint (and optionally train logL per timepoint)
#
# Requires (from prior cells):
#   - SCANNED_PROTOCOLS, models_index_df
#   - X_all, lengths_all, sequences_all_df
#   - UNITS_DF and unit->sequence mapping helpers:
#       * UNIT_ID_TO_SEQ_INDICES  (dict)
#       * SEQ_LOOKUP              (dict: seq_index -> dict(row_start,row_end,length))
#   - CV_SPLITS (list of dicts)
#   - YAML-derived dicts:
#       * MINIMUMS_CFG, GENERALIZATION_CFG
#   - HMM_DEFAULTS, HMM_PRESETS
#
# Produces:
#   - GENERALIZATION_RESULTS_LONG_DF
#   - GENERALIZATION_SUMMARY_DF
#
# Writes:
#   - .../_cross_model_evaluation/generalization_summary.csv
#   - .../_cross_model_evaluation/generalization_details.csv (optional)
# __________________________________________________________________________________________________________


print("\n" + "=" * 104)
print("[GENERALIZATION] Running K-fold generalization evaluation")
print("=" * 104)

# Set basic guardrails:
if not bool(CROSS_MODEL_EVAL):
    print("[GENERALIZATION] cross_model_evaluation.enabled=False -> skipping.")
    GENERALIZATION_RESULTS_LONG_DF = pd.DataFrame()
    GENERALIZATION_SUMMARY_DF = pd.DataFrame()
else:
    if not bool(GENERALIZATION_CFG.get("enabled", False)):
        print("[GENERALIZATION] generalization.enabled=False -> skipping.")
        GENERALIZATION_RESULTS_LONG_DF = pd.DataFrame()
        GENERALIZATION_SUMMARY_DF = pd.DataFrame()
    else:
        if "CV_SPLITS" not in globals() or not CV_SPLITS:
            raise RuntimeError("[GENERALIZATION ERROR] CV_SPLITS is missing/empty. Run the split-building cell first.")

        if "SCANNED_PROTOCOLS" not in globals() or not SCANNED_PROTOCOLS:
            raise RuntimeError("[GENERALIZATION ERROR] SCANNED_PROTOCOLS missing/empty. Run the scanner cell first.")

        if "UNIT_ID_TO_SEQ_INDICES" not in globals() or not UNIT_ID_TO_SEQ_INDICES:
            raise RuntimeError("[GENERALIZATION ERROR] UNIT_ID_TO_SEQ_INDICES missing/empty. Run the unit-materialization cell first.")

        if "SEQ_LOOKUP" not in globals() or not SEQ_LOOKUP:
            raise RuntimeError("[GENERALIZATION ERROR] SEQ_LOOKUP missing/empty. Run the unit-materialization cell first.")

        # Set output directory for cross-model eval artifacts:
        CROSS_EVAL_DIR = Path(DATASET_MODELS_DIR) / "_cross_model_evaluation"
        CROSS_EVAL_DIR.mkdir(parents=True, exist_ok=True)

        # Resolve generalization-testing parameters:
        n_folds   = int(GENERALIZATION_CFG.get("n_folds", 5))
        n_repeats = int(GENERALIZATION_CFG.get("n_repeats", 1))
        stratify = bool(GENERALIZATION_CFG.get("stratify_by_group", False))  # currently not used unless you later add group labels

        metric_cfg = GENERALIZATION_CFG.get("metrics", {}) or {}
        want_test_ll_pt  = bool(metric_cfg.get("test_logL_per_timepoint", True))
        want_train_ll_pt = bool(metric_cfg.get("train_logL_per_timepoint", False))

        report_cfg = GENERALIZATION_CFG.get("report", {}) or {}
        save_per_fold_details = bool(report_cfg.get("save_per_fold_details", True))

        print("[GENERALIZATION] Settings:")
        print(f"   n_folds   = {n_folds}")
        print(f"   n_repeats = {n_repeats}")
        print(f"   stratify_by_group = {stratify} (not used unless group labels wired in)")
        print(f"   metrics: test_logL_per_timepoint={want_test_ll_pt}, train_logL_per_timepoint={want_train_ll_pt}")
        print(f"   save_per_fold_details={save_per_fold_details}")

        # ----------------------------
        # Minimal internal helpers:
        # ----------------------------
        # We need a reliable way to extract train/test matrices from unit ids:
        if "_seq_indices_for_units" not in globals():
            def _seq_indices_for_units(unit_ids):
                seqs = []
                for u in unit_ids:
                    if u not in UNIT_ID_TO_SEQ_INDICES:
                        continue
                    seqs.extend(list(UNIT_ID_TO_SEQ_INDICES[u]))
                return sorted(set(seqs))

        if "_build_X_and_lengths_from_seq_indices" not in globals():
            def _build_X_and_lengths_from_seq_indices(seq_indices):
                """
                Build X and lengths by concatenating the already-ordered slices in 'X_all'
                according to 'SEQ_LOOKUP' row_start/row_end.
                """
                if not seq_indices:
                    X = np.empty((0, int(X_all.shape[1])), dtype=float)
                    lengths = []
                    return X, lengths

                chunks = []
                lengths = []
                for sidx in seq_indices:
                    meta = SEQ_LOOKUP.get(int(sidx), None)
                    if meta is None:
                        continue
                    rs = int(meta["row_start"])
                    re = int(meta["row_end"])
                    L  = int(meta["length"])
                    if L <= 0:
                        continue
                    chunks.append(X_all[rs:re + 1, :])
                    lengths.append(L)

                if not chunks:
                    X = np.empty((0, int(X_all.shape[1])), dtype=float)
                    lengths = []
                    return X, lengths

                X = np.vstack(chunks)
                return X, lengths

        # Iterate protocols & perform evaluation:
        seed_is_int = isinstance(RANDOM_SEED, (int, np.integer))
        base_seed = int(RANDOM_SEED) if seed_is_int else None

        long_rows = []
        summary_rows = []

        # Grab protocols in the scan index (keeping stable ordering for output):
        protocols_ordered = list(models_index_df["protocol"].tolist()) if "models_index_df" in globals() else sorted(SCANNED_PROTOCOLS.keys())

        for protocol_name in protocols_ordered:
            if protocol_name not in SCANNED_PROTOCOLS:
                continue

            info = SCANNED_PROTOCOLS[protocol_name]
            payload = info.get("chosen_payload", {}) if isinstance(info.get("chosen_payload", None), dict) else {}

            chosen_k = payload.get("chosen_K", None)
            if chosen_k is None:
                handle_error(f"[GENERALIZATION WARN] Protocol '{protocol_name}': chosen_K missing in chosen_model.json; skipping.")
                continue
            chosen_k = int(chosen_k)

            # Resolve effective protocol config from YAML presets & defaults:
            if protocol_name not in HMM_PRESETS:
                handle_error(f"[GENERALIZATION WARN] Protocol '{protocol_name}' not found in HMM_protocols.presets; skipping.")
                continue

            proto_cfg = dict(HMM_DEFAULTS)
            proto_cfg.update(dict(HMM_PRESETS[protocol_name]))

            cov_type = str(proto_cfg.get("covariance_type", "diag")).strip().lower()
            max_iter = int(proto_cfg.get("max_iterations", 500))
            tol      = float(proto_cfg.get("convergence_tolerance", 1e-4))

            pca_cfg = proto_cfg.get("PCA", {}) or {}
            pca_enabled     = bool(pca_cfg.get("enabled", False))
            pca_n_components = pca_cfg.get("n_components", None)
            pca_whitening   = bool(pca_cfg.get("whitening", False))

            print("\n" + "-" * 104)
            print(f"[GENERALIZATION] Protocol: {protocol_name}  |  K={chosen_k}  |  cov='{cov_type}'  |  PCA={pca_enabled}")
            print("-" * 104)

            # Initialize fold-level accumulators:
            test_ll_pt_vals = []
            train_ll_pt_vals = []

            n_folds_run = 0
            n_folds_skipped = 0

            for split in CV_SPLITS:
                # Robustly resolve unit lists from split dictionaries:
                train_units = (
                    split.get("train_unit_ids",
                              split.get("train_units",
                                        split.get("A_unit_ids", None))))
                test_units = (
                    split.get("test_unit_ids",
                              split.get("test_units",
                                        split.get("B_unit_ids", None))))

                if train_units is None or test_units is None:
                    handle_error(f"[GENERALIZATION WARN] Missing train/test unit ids in split keys: {list(split.keys())}. Skipping split.")
                    n_folds_skipped += 1
                    continue

                # Convert to lists (e.g. they may be numpy arrays):
                train_units = list(train_units)
                test_units = list(test_units)

                # Enforce minimums (if configured):
                min_seq = int(MINIMUMS_CFG.get("min_sequences_per_fold", 1))
                min_obs = int(MINIMUMS_CFG.get("min_total_observations_per_fold", 1))
                min_len = int(MINIMUMS_CFG.get("min_sequence_length", 1))

                seq_train = _seq_indices_for_units(train_units)
                seq_test = _seq_indices_for_units(test_units)

                # Materialize (X, lengths):
                X_train_fold, lengths_train_fold = _build_X_and_lengths_from_seq_indices(seq_train)
                X_test_fold, lengths_test_fold = _build_X_and_lengths_from_seq_indices(seq_test)

                # Apply minimum sequence length filter:
                def _filter_by_min_len(seq_indices):
                    keep = []
                    for sidx in seq_indices:
                        meta = SEQ_LOOKUP.get(int(sidx), None)
                        if meta is None:
                            continue
                        if int(meta.get("length", 0)) >= int(min_len):
                            keep.append(int(sidx))
                    return keep

                seq_train = _filter_by_min_len(seq_train)
                seq_test = _filter_by_min_len(seq_test)

                # Re-materialize after filtering:
                X_train_fold, lengths_train_fold = _build_X_and_lengths_from_seq_indices(seq_train)
                X_test_fold, lengths_test_fold = _build_X_and_lengths_from_seq_indices(seq_test)

                if len(seq_train) < min_seq or len(seq_test) < 1:
                    n_folds_skipped += 1
                    continue
                if int(X_train_fold.shape[0]) < min_obs or int(X_test_fold.shape[0]) < 1:
                    n_folds_skipped += 1
                    continue
                if int(np.sum(lengths_train_fold)) != int(X_train_fold.shape[0]) or int(np.sum(lengths_test_fold)) != int(X_test_fold.shape[0]):
                    handle_error("[GENERALIZATION WARN] Fold materialization produced mismatch between X rows and lengths sums; skipping fold.")
                    n_folds_skipped += 1
                    continue

                # -----------------------------------
                # Fit PCA on training fold & transform:
                # -----------------------------------
                pca_obj = None
                X_train_eff = X_train_fold
                X_test_eff = X_test_fold

                if pca_enabled:
                    pca_obj = PCA(
                        n_components=pca_n_components,
                        whiten=pca_whitening,
                        random_state=base_seed)
                    X_train_eff = pca_obj.fit_transform(X_train_fold)
                    X_test_eff = pca_obj.transform(X_test_fold)

                # -----------------------------------
                # Fit model on training fold:
                # -----------------------------------
                # Keep the random_state stable per fold, but deterministic
                fold_id = int(split.get("fold", split.get("k", 0)))
                rep_id = int(split.get("repeat", 0))
                rs = (base_seed + 1000 * rep_id + fold_id) if seed_is_int else None

                model = GaussianHMM(
                    n_components=int(chosen_k),
                    covariance_type=cov_type,
                    n_iter=int(max_iter),
                    tol=float(tol),
                    random_state=rs,
                    verbose=False)

                try:
                    model.fit(X_train_eff, lengths_train_fold)
                except Exception as exc:
                    handle_error(f"[GENERALIZATION WARN] Fit failed for protocol='{protocol_name}', fold={fold_id}, repeat={rep_id}: {exc}")
                    n_folds_skipped += 1
                    continue

                # -----------------------------------
                # Score test (& optionally train):
                # -----------------------------------
                test_ll = np.nan
                test_ll_pt = np.nan
                train_ll = np.nan
                train_ll_pt = np.nan

                try:
                    test_ll = float(model.score(X_test_eff, lengths_test_fold))
                    test_ll_pt = float(test_ll / max(1, int(X_test_eff.shape[0])))
                except Exception as exc:
                    handle_error(f"[GENERALIZATION WARN] Test score failed for protocol='{protocol_name}', fold={fold_id}, repeat={rep_id}: {exc}")

                if want_train_ll_pt:
                    try:
                        train_ll = float(model.score(X_train_eff, lengths_train_fold))
                        train_ll_pt = float(train_ll / max(1, int(X_train_eff.shape[0])))
                    except Exception as exc:
                        handle_error(f"[GENERALIZATION WARN] Train score failed for protocol='{protocol_name}', fold={fold_id}, repeat={rep_id}: {exc}")

                # Record:
                n_folds_run += 1
                if want_test_ll_pt:
                    test_ll_pt_vals.append(test_ll_pt)
                if want_train_ll_pt:
                    train_ll_pt_vals.append(train_ll_pt)

                long_rows.append({
                    "protocol": protocol_name,
                    "K": int(chosen_k),
                    "covariance_type": cov_type,
                    "pca_enabled": bool(pca_enabled),
                    "pca_n_components": pca_n_components,
                    "pca_whitening": bool(pca_whitening),
                    "repeat": int(rep_id),
                    "fold": int(fold_id),
                    "n_train_units": int(len(train_units)),
                    "n_test_units": int(len(test_units)),
                    "n_train_sequences": int(len(seq_train)),
                    "n_test_sequences": int(len(seq_test)),
                    "n_train_obs": int(X_train_eff.shape[0]),
                    "n_test_obs": int(X_test_eff.shape[0]),
                    "test_logL": float(test_ll) if not np.isnan(test_ll) else np.nan,
                    "test_logL_per_timepoint": float(test_ll_pt) if not np.isnan(test_ll_pt) else np.nan,
                    "train_logL": float(train_ll) if not np.isnan(train_ll) else np.nan,
                    "train_logL_per_timepoint": float(train_ll_pt) if not np.isnan(train_ll_pt) else np.nan})

            # ----------------------------
            # Summarize protocol:
            # ----------------------------
            test_mean = float(np.nanmean(test_ll_pt_vals)) if test_ll_pt_vals else np.nan
            test_std  = float(np.nanstd(test_ll_pt_vals)) if test_ll_pt_vals else np.nan

            train_mean = float(np.nanmean(train_ll_pt_vals)) if train_ll_pt_vals else np.nan
            train_std  = float(np.nanstd(train_ll_pt_vals)) if train_ll_pt_vals else np.nan

            print(f"[GENERALIZATION] Protocol '{protocol_name}' complete.")
            print(f"   folds run    = {n_folds_run} (skipped: {n_folds_skipped})")
            if want_test_ll_pt:
                print(f"   test_logL/TP = mean={test_mean:.3f}, std={test_std:.3f}")
            if want_train_ll_pt:
                print(f"   train_logL/TP= mean={train_mean:.3f}, std={train_std:.3f}")

            summary_rows.append({
                "protocol": protocol_name,
                "K": int(chosen_k),
                "covariance_type": cov_type,
                "pca_enabled": bool(pca_enabled),
                "pca_n_components": pca_n_components,
                "pca_whitening": bool(pca_whitening),
                "n_folds_run": int(n_folds_run),
                "n_folds_skipped": int(n_folds_skipped),
                "test_logL_per_timepoint_mean": test_mean,
                "test_logL_per_timepoint_std": test_std,
                "train_logL_per_timepoint_mean": train_mean if want_train_ll_pt else np.nan,
                "train_logL_per_timepoint_std": train_std if want_train_ll_pt else np.nan,
                "final_model_path": info.get("final_model_path", None)})

        # ----------------------------
        # Build DataFrames & write outputs:
        # ----------------------------
        GENERALIZATION_RESULTS_LONG_DF = pd.DataFrame(long_rows)
        GENERALIZATION_SUMMARY_DF = pd.DataFrame(summary_rows).sort_values(["protocol"]).reset_index(drop=True)

        print("\n[GENERALIZATION] Per-protocol generalization summary:")
        display(GENERALIZATION_SUMMARY_DF)

        summary_path = CROSS_EVAL_DIR / "generalization_summary.csv"
        GENERALIZATION_SUMMARY_DF.to_csv(summary_path, index=False)
        print(f"\n[GENERALIZATION] Wrote: {summary_path}")

        if save_per_fold_details:
            details_path = CROSS_EVAL_DIR / "generalization_details.csv"
            GENERALIZATION_RESULTS_LONG_DF.to_csv(details_path, index=False)
            print(f"[GENERALIZATION] Wrote: {details_path}")
        else:
            print("[GENERALIZATION] save_per_fold_details=False -> fold-level CSV not written (in-memory DF still available as GENERALIZATION_RESULTS_LONG_DF).")

**Notes on the above metrics / outputs:**

- **test_logL_per_timepoint_mean**: Higher is better, though scores can't be objectively compared across models.
- **test_logL_per_timepoint_std**: Lower is better; indicates LESS variability across training folds.

---------

--------
--------
### **Placeholder:** Visualization suite

-QC plots for above stability & generalization results, etc.

--------
--------

## **Final (manual) model selection:**

In [ ]:
# __________________________________________________________________________________________________________
### FINAL-FINAL MODEL SELECTION (MANUAL, ONE-LINE EDIT)
# __________________________________________________________________________________________________________

# Basic checks:
if "models_index_df" not in globals():
    raise RuntimeError("[FINAL SELECT ERROR] Missing 'models_index_df'. Run the scanner cell first.")

# Expected columns (schemas):
STAB_COLS = ["protocol", "K", "occupancy_corr_mean", "transition_frobenius_mean", "dwell_jsd_mean"]
GEN_COLS  = ["protocol", "K", "test_logL_per_timepoint_mean", "test_logL_per_timepoint_std"]

def _ensure_df_schema(df, required_cols):
    """
    Ensure df exists and has required columns.
    If df is missing/None or missing all required columns, return a new empty df with required_cols.
    If df exists but is missing some required columns, add them as NaN.
    """
    if df is None or not isinstance(df, pd.DataFrame):
        return pd.DataFrame(columns=required_cols)

    # If df has none of the required columns, treat it as incompatible (likely an empty placeholder) and reset schema.
    if len(set(required_cols).intersection(set(df.columns))) == 0:
        return pd.DataFrame(columns=required_cols)

    # Otherwise, add any missing required columns.
    out = df.copy()
    for c in required_cols:
        if c not in out.columns:
            out[c] = np.nan
    return out

# Ensure stability/generalization DFs are merge-safe (even if skipped earlier):
STABILITY_SUMMARY_DF = _ensure_df_schema(globals().get("STABILITY_SUMMARY_DF", None), STAB_COLS)
GENERALIZATION_SUMMARY_DF = _ensure_df_schema(globals().get("GENERALIZATION_SUMMARY_DF", None), GEN_COLS)

# Merge a compact review table for user decision-making
review = (
    models_index_df[["protocol"]]
    .merge(STABILITY_SUMMARY_DF[STAB_COLS], on="protocol", how="left", suffixes=("", "_stab"))
    .merge(GENERALIZATION_SUMMARY_DF[GEN_COLS], on="protocol", how="left", suffixes=("", "_gen")))

# Fill K from scanner metadata when evaluation metrics are missing (e.g. if 'CROSS_MODEL_EVAL' is disabled):
if "chosen_K" in models_index_df.columns:
    review = review.merge(models_index_df[["protocol", "chosen_K"]], on="protocol", how="left")
    if "K" in review.columns:
        review["K"] = review["K"].where(review["K"].notna(), review["chosen_K"])
    else:
        review["K"] = review["chosen_K"]
    review = review.drop(columns=["chosen_K"])

review = review.sort_values(["protocol"]).reset_index(drop=True)

print("\n[FINAL SELECT] Candidate protocols:")
display(review.drop(columns=['K_gen']))

----------
### **User input required here:**

**Note:** It's generally advisable to delete / clear out one's manual input immediately after running this cell, so you don't mistakenly carry over old selections to new eval conditions.

In [ ]:
######################################################################
#   ================= USER ACTION REQUIRED HERE =================
######################################################################
######################################################################

# Type ONE protocol name exactly as shown in the 'protocol' column above (in single quotes):

FINAL_MODEL_SELECTION = ''

######################################################################
######################################################################
######################################################################

if FINAL_MODEL_SELECTION is None or str(FINAL_MODEL_SELECTION).strip() == "":
    raise RuntimeError("[FINAL SELECT ERROR] Please set FINAL_MODEL_SELECTION and re-run this cell.")
FINAL_MODEL_SELECTION = str(FINAL_MODEL_SELECTION).strip()
FINAL_MODEL_PROTOCOL = FINAL_MODEL_SELECTION  # <-- critical wiring for downstream export
if FINAL_MODEL_SELECTION not in set(review["protocol"].tolist()):
    raise RuntimeError(
        f"[FINAL SELECT ERROR] FINAL_MODEL_SELECTION='{FINAL_MODEL_SELECTION}' not found.\n"
        f"Available: {sorted(set(review['protocol'].tolist()))}")
FINAL_MODEL_K = int(review.loc[review["protocol"] == FINAL_MODEL_SELECTION, "K"].iloc[0])
print("\n[FINAL MODEL SELECTION:]")
print(f"   FINAL_MODEL_PROTOCOL   = {FINAL_MODEL_PROTOCOL}")
print(f"   FINAL_MODEL_K          = {FINAL_MODEL_K}")

-----------

**Next step:** One final fresh re-fitting of the model (i.e. just in case 'refit_on_all' parameter was not previously enabled):

In [ ]:
# __________________________________________________________________________________________________________
### FINAL MODEL EXPORT (ALWAYS RE-FIT ON X_all)
#
# Writes a single canonical bundle to:
#   DATASET_MODELS_DIR / "FINAL_MODEL" /
#       final_model.joblib
#       final_pca.joblib               (only if PCA enabled)
#       final_model_selection.json
#       feature_columns.json           (recommended)
#
# Requires:
#   - FINAL_MODEL_PROTOCOL, FINAL_MODEL_K
#   - X_all, lengths_all
#   - feature_columns_final (from your X_all reconstruction cell)
#   - HMM_PRESETS, HMM_DEFAULTS
#   - DATASET_MODELS_DIR, DATASET_NAME
# __________________________________________________________________________________________________________

# Quick sanity-checks:
required = ["FINAL_MODEL_PROTOCOL", "FINAL_MODEL_K", "X_all", "lengths_all", "HMM_PRESETS", "HMM_DEFAULTS", "DATASET_MODELS_DIR", "DATASET_NAME"]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(f"[FINAL EXPORT ERROR] Missing required variable(s): {missing}")

if "feature_columns_final" not in globals() or not feature_columns_final:
    handle_error("[FINAL EXPORT WARN] feature_columns_final missing/empty; feature_columns.json will not be written.")

proto_name = str(FINAL_MODEL_PROTOCOL).strip()
if proto_name not in HMM_PRESETS:
    raise RuntimeError(
        f"[FINAL EXPORT ERROR] Protocol '{proto_name}' not found in config['HMM_protocols']['presets']. "
        f"Available: {sorted(list(HMM_PRESETS.keys()))}")

# Resolve effective protocol config using defaults:
proto_raw = dict(HMM_PRESETS[proto_name])
def _fallback(v, default):
    return default if (v is None or (isinstance(v, str) and v.strip() == "")) else v

cov_type = str(proto_raw.get("covariance_type", "diag")).strip().lower()

pca_cfg = proto_raw.get("PCA", {}) or {}
pca_enabled = bool(pca_cfg.get("enabled", False))
pca_n_components = pca_cfg.get("n_components", None)
pca_whitening = bool(pca_cfg.get("whitening", False))

max_iter = int(_fallback(proto_raw.get("max_iterations", None), HMM_DEFAULTS.get("max_iterations", 500)))
tol      = float(_fallback(proto_raw.get("convergence_tolerance", None), HMM_DEFAULTS.get("convergence_tolerance", 1e-4)))

# Warn on questionable parameter combinations (but do not block):
if cov_type == "full" and not pca_enabled:
    print("[WARNING:] Selected protocol uses covariance_type='full' with PCA disabled; this may be slow/unstable in high dimensions.")

# Set output folder:
FINAL_DIR = Path(DATASET_MODELS_DIR) / "FINAL_MODEL"

# Overwrite guard:
#   --> If 'FINAL_DIR' already contains a prior bundle, refuse to overwrite unless explicitly allowed:
prior_bundle_files = [
    FINAL_DIR / "final_model.joblib",
    FINAL_DIR / "final_pca.joblib",
    FINAL_DIR / "final_model_selection.json",
    FINAL_DIR / "feature_columns.json"]
bundle_exists = any(p.exists() for p in prior_bundle_files)
if bundle_exists and (not OVERWRITE_FINAL_MODEL):
    existing = [p.name for p in prior_bundle_files if p.exists()]
    raise RuntimeError(
        "[FINAL EXPORT ERROR] FINAL_MODEL bundle already exists and "
        "cross_model_evaluation.overwrite_final_model=False.\n"
        f"  FINAL_DIR = {FINAL_DIR}\n"
        f"  Existing files = {existing}\n"
        "Set cross_model_evaluation.overwrite_final_model: true in config.yaml to overwrite.")
if bundle_exists and OVERWRITE_FINAL_MODEL:
    existing = [p.name for p in prior_bundle_files if p.exists()]
    print(
        "[FINAL EXPORT] overwrite_final_model=True --> existing FINAL_MODEL bundle will be overwritten.\n"
        f"  FINAL_DIR = {FINAL_DIR}\n"
        f"  Existing files = {existing}")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Always re-fit preprocessing on 'X_all' (if any):
X_eff = X_all
pca_obj = None

seed_is_int = isinstance(RANDOM_SEED, (int, np.integer))
base_seed = int(RANDOM_SEED) if seed_is_int else None

if pca_enabled:
    print(f"[FINAL EXPORT] Refitting PCA on X_all: n_components={pca_n_components}, whitening={pca_whitening}")
    pca_obj = PCA(
        n_components=pca_n_components,
        whiten=pca_whitening,
        random_state=base_seed)
    X_eff = pca_obj.fit_transform(X_all)
    joblib.dump(pca_obj, FINAL_DIR / "final_pca.joblib")
    print(f"[FINAL EXPORT] Wrote: {FINAL_DIR / 'final_pca.joblib'}")

# Fit the FINAL-final model on X_all:
print(f"[FINAL EXPORT] Fitting GaussianHMM on X_all: K={int(FINAL_MODEL_K)}, cov='{cov_type}', max_iter={max_iter}, tol={tol}")
final_model = GaussianHMM(
    n_components=int(FINAL_MODEL_K),
    covariance_type=cov_type,
    n_iter=max_iter,
    tol=tol,
    random_state=base_seed,
    verbose=False)
final_model.fit(X_eff, lengths_all)

# Save model:
final_model_path = FINAL_DIR / "final_model.joblib"
joblib.dump(final_model, final_model_path)
print(f"[FINAL EXPORT] Wrote: {final_model_path}")

# Pull metrics from summary dataframes:
stab_row = None
gen_row = None
try:
    stab_row = STABILITY_SUMMARY_DF.loc[STABILITY_SUMMARY_DF["protocol"] == proto_name].iloc[0].to_dict()
except Exception:
    stab_row = None
try:
    gen_row = GENERALIZATION_SUMMARY_DF.loc[GENERALIZATION_SUMMARY_DF["protocol"] == proto_name].iloc[0].to_dict()
except Exception:
    gen_row = None

# Write selection/provenance JSON sidecar:
selection_payload = {
    "timestamp_unix": int(time.time()),
    "dataset_name": str(DATASET_NAME),
    "final_dir": str(FINAL_DIR),
    "selected_protocol": proto_name,
    "selected_K": int(FINAL_MODEL_K),
    "refit_on_X_all_always": True,
    "protocol_config_effective": {
        "covariance_type": cov_type,
        "PCA": {
            "enabled": bool(pca_enabled),
            "n_components": pca_n_components,
            "whitening": bool(pca_whitening)},
        "max_iterations": int(max_iter),
        "convergence_tolerance": float(tol),
        "random_seed_used": base_seed},
    "artifacts": {
        "final_model_joblib": str(final_model_path),
        "final_pca_joblib": str(FINAL_DIR / "final_pca.joblib") if pca_enabled else None},
    "summary_metrics": {
        "stability_row": stab_row,
        "generalization_row": gen_row}}

with open(FINAL_DIR / "final_model_selection.json", "w") as f:
    json.dump(selection_payload, f, indent=2)
print(f"[FINAL EXPORT] Wrote: {FINAL_DIR / 'final_model_selection.json'}")

# Optional: write feature columns for downstream safety:
if "feature_columns_final" in globals() and feature_columns_final:
    with open(FINAL_DIR / "feature_columns.json", "w") as f:
        json.dump({"feature_columns": list(feature_columns_final)}, f, indent=2)
    print(f"[FINAL EXPORT] Wrote: {FINAL_DIR / 'feature_columns.json'}")

print("\n[FINAL EXPORT] Final model bundle is ready for downstream decoding & feature-generation:")
print(f"   FINAL_DIR = {FINAL_DIR}")

---------